# 01_Ingestion

DiVA OAI-PMH harvest: raw XML pages + manifest under Files/diva/raw/<run_id>/.

Generated from `src/diva_lakehouse` by `scripts/build_notebooks.py`. Attach a default Lakehouse before running. See `docs/setup-fabric.md`.

In [ ]:
# =============================================================================
# DiVA Research Lakehouse - 01_Ingestion
# GENERATED by scripts/build_notebooks.py from src/diva_lakehouse - do not edit here.
# Requires: a default Lakehouse attached to this notebook (see docs/setup-fabric.md).
# =============================================================================
# Parameters: edit here, then Run all.

source_mode = "live"  # "live" = DiVA OAI-PMH (normal) | "demo" = SYNTHETIC fixtures (explicit choice)
oai_endpoint = "https://www.diva-portal.org/dice/oai"  # checked with Identify before harvesting

# OAI-PMH from/until filter on the SOURCE DATESTAMP (when DiVA created/changed the
# record), NOT on the thesis publication year. Historical default window below.
# from_datestamp = "auto" continues from the last complete + processed interval.
from_datestamp = "2024-03-01"
until_datestamp = "2024-03-02"
set_spec = ""  # optional OAI set (setSpec); "" = no set restriction

max_records = 500  # harvest stops here; the interval is then marked INCOMPLETE
table_prefix = ""  # "" -> "diva" (live) / "diva_demo" (demo); live and demo never share tables
run_id = ""  # "" -> new run id (00/01) or the latest started run (02-04)
run_mode = "pipeline"  # "pipeline" | "rerun_demo" (00_Run_All only, needs source_mode="demo")
demo_batch = "A"  # demo + pipeline only: "A" or "B" (SYNTHETIC batches)

In [ ]:
# Embedded diva_lakehouse package (generated). Registers an import hook so the
# modules in the following cells can be imported like a normal package.
import importlib.abc
import importlib.util
import sys

_DIVA_SOURCES = globals().get("_DIVA_SOURCES", {})


class _DivaEmbeddedFinder(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path, target=None):
        if fullname in _DIVA_SOURCES:
            return importlib.util.spec_from_loader(fullname, self, is_package=(fullname == "diva_lakehouse"))
        return None

    def create_module(self, spec):
        return None

    def exec_module(self, module):
        if module.__name__ == "diva_lakehouse":
            module.__path__ = []
        code = compile(_DIVA_SOURCES[module.__name__], f"<diva_lakehouse:{module.__name__}>", "exec")
        exec(code, module.__dict__)


sys.meta_path[:] = [f for f in sys.meta_path if type(f).__name__ != "_DivaEmbeddedFinder"]
sys.meta_path.insert(0, _DivaEmbeddedFinder())


def _diva_register(name, source):
    _DIVA_SOURCES[name] = source
    for loaded in [m for m in sys.modules if m == "diva_lakehouse" or m.startswith("diva_lakehouse.")]:
        del sys.modules[loaded]  # re-running the cells always uses the embedded code

In [ ]:
# Embedded module diva_lakehouse (generated from src/diva_lakehouse/__init__.py)
_diva_register('diva_lakehouse', r'''"""DiVA Research Lakehouse: OAI-PMH metadata -> Bronze/Silver/Gold Delta tables.

The package is pure Python apart from the Spark/Delta stages (bronze, silver,
gold, quality, pipeline). The Fabric notebooks embed this package verbatim; see
scripts/build_notebooks.py.
"""

__version__ = "0.1.0"
''')

In [ ]:
# Embedded module diva_lakehouse.config (generated from src/diva_lakehouse/config.py)
_diva_register('diva_lakehouse.config', r'''"""Pipeline configuration, validation and naming of tables and paths."""

from __future__ import annotations

import re
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone

# Candidate endpoint. It could NOT be verified from the build environment (see
# docs/setup-fabric.md). 01_Ingestion always calls Identify first and stops with
# a clear message if the endpoint does not answer as an OAI-PMH 2.0 repository.
DEFAULT_OAI_ENDPOINT = "https://www.diva-portal.org/dice/oai"

# Default harvest window: a short, historical, explicitly stated interval.
# NOTE: these are OAI-PMH *datestamps* (when the record was created/changed in
# DiVA), not the thesis publication year.
DEFAULT_FROM_DATESTAMP = "2024-03-01"
DEFAULT_UNTIL_DATESTAMP = "2024-03-02"

DEFAULT_MAX_RECORDS = 500
HARD_MAX_RECORDS = 1000

LIVE_TABLE_PREFIX = "diva"
DEMO_TABLE_PREFIX = "diva_demo"
LIVE_FILES_BASE = "diva"
DEMO_FILES_BASE = "diva/demo"

SOURCE_MODES = ("live", "demo")
RUN_MODES = ("pipeline", "rerun_demo")
THESIS_FILTERS = ("student_thesis", "none")

_DATESTAMP_RE = re.compile(r"^\d{4}-\d{2}-\d{2}(T\d{2}:\d{2}:\d{2}Z)?$")
_RUN_ID_RE = re.compile(r"^[A-Za-z0-9][A-Za-z0-9_.-]{0,79}$")
_PREFIX_RE = re.compile(r"^[a-z][a-z0-9_]{0,39}$")


class ConfigError(ValueError):
    """Raised when notebook parameters are invalid."""


def utc_now() -> datetime:
    return datetime.now(timezone.utc).replace(microsecond=0)


def iso_utc(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def new_run_id(source_mode: str, now: datetime | None = None) -> str:
    now = now or utc_now()
    return f"{source_mode}-{now.strftime('%Y%m%dT%H%M%SZ')}"


@dataclass
class PipelineConfig:
    source_mode: str = "live"
    oai_endpoint: str = DEFAULT_OAI_ENDPOINT
    metadata_prefix: str = "oai_dc"
    # "auto" = continue from the last fully harvested AND processed interval.
    from_datestamp: str = DEFAULT_FROM_DATESTAMP
    until_datestamp: str = DEFAULT_UNTIL_DATESTAMP
    set_spec: str = ""
    max_records: int = DEFAULT_MAX_RECORDS
    max_pages: int = 50
    table_prefix: str = ""  # "" -> diva (live) / diva_demo (demo)
    run_id: str = ""  # "" -> generated (01/00) or latest run (02-04)
    run_mode: str = "pipeline"  # pipeline | rerun_demo
    demo_batch: str = "A"  # demo + pipeline only: A or B
    thesis_filter: str = "student_thesis"
    reuse_existing_raw: bool = True
    request_timeout_s: float = 60.0
    max_retries: int = 3
    backoff_base_s: float = 2.0
    polite_delay_s: float = 1.0
    low_selection_threshold: int = 10
    lakehouse_root: str = "auto"  # "auto" -> /lakehouse/default (Fabric mount)
    user_agent: str = "diva-research-lakehouse/0.1 (portfolio project; OAI-PMH harvester)"
    extra: dict = field(default_factory=dict)

    def __post_init__(self) -> None:
        self.source_mode = (self.source_mode or "").strip().lower()
        self.run_mode = (self.run_mode or "").strip().lower()
        self.thesis_filter = (self.thesis_filter or "").strip().lower()
        self.set_spec = (self.set_spec or "").strip()
        self.run_id = (self.run_id or "").strip()
        self.demo_batch = (self.demo_batch or "").strip().upper()
        self.from_datestamp = (self.from_datestamp or "").strip()
        self.until_datestamp = (self.until_datestamp or "").strip()
        if not self.table_prefix:
            self.table_prefix = DEMO_TABLE_PREFIX if self.source_mode == "demo" else LIVE_TABLE_PREFIX
        self.table_prefix = self.table_prefix.strip().lower()
        self.validate()

    # ------------------------------------------------------------------ checks
    def validate(self) -> None:
        if self.source_mode not in SOURCE_MODES:
            raise ConfigError(f"source_mode must be one of {SOURCE_MODES}, got {self.source_mode!r}")
        if self.run_mode not in RUN_MODES:
            raise ConfigError(f"run_mode must be one of {RUN_MODES}, got {self.run_mode!r}")
        if self.run_mode == "rerun_demo" and self.source_mode != "demo":
            raise ConfigError("run_mode='rerun_demo' requires source_mode='demo' (it resets the demo tables).")
        if self.thesis_filter not in THESIS_FILTERS:
            raise ConfigError(f"thesis_filter must be one of {THESIS_FILTERS}")
        if self.source_mode == "demo" and self.demo_batch not in ("A", "B"):
            raise ConfigError("demo_batch must be 'A' or 'B'")
        if not isinstance(self.max_records, int) or not 1 <= self.max_records <= HARD_MAX_RECORDS:
            raise ConfigError(f"max_records must be an int between 1 and {HARD_MAX_RECORDS}")
        if not isinstance(self.max_pages, int) or self.max_pages < 1:
            raise ConfigError("max_pages must be a positive int")
        if not _PREFIX_RE.match(self.table_prefix):
            raise ConfigError("table_prefix must match ^[a-z][a-z0-9_]{0,39}$")
        # Live and demo data must never share tables.
        is_demo_prefix = "demo" in self.table_prefix
        if self.source_mode == "demo" and not is_demo_prefix:
            raise ConfigError("In demo mode table_prefix must contain 'demo' (keeps demo and live data apart).")
        if self.source_mode == "live" and is_demo_prefix:
            raise ConfigError("In live mode table_prefix must not contain 'demo'.")
        if self.run_id and not _RUN_ID_RE.match(self.run_id):
            raise ConfigError("run_id may only contain letters, digits, '_', '-', '.' (max 80 chars)")
        if self.source_mode == "live":
            if not self.oai_endpoint.startswith(("https://", "http://")):
                raise ConfigError("oai_endpoint must be an http(s) URL")
            if self.from_datestamp != "auto" and not _DATESTAMP_RE.match(self.from_datestamp):
                raise ConfigError("from_datestamp must be YYYY-MM-DD, YYYY-MM-DDThh:mm:ssZ or 'auto'")
            if not _DATESTAMP_RE.match(self.until_datestamp):
                raise ConfigError("until_datestamp must be YYYY-MM-DD or YYYY-MM-DDThh:mm:ssZ")
            if self.from_datestamp != "auto":
                if len(self.from_datestamp) != len(self.until_datestamp):
                    raise ConfigError("from_datestamp and until_datestamp must use the same granularity")
                if self.from_datestamp > self.until_datestamp:
                    raise ConfigError("from_datestamp must be <= until_datestamp")
        for name in ("request_timeout_s", "backoff_base_s", "polite_delay_s"):
            if float(getattr(self, name)) < 0:
                raise ConfigError(f"{name} must be >= 0")
        if self.max_retries < 0 or self.max_retries > 6:
            raise ConfigError("max_retries must be between 0 and 6")

    # ----------------------------------------------------------------- naming
    @property
    def files_base(self) -> str:
        """Base folder under the Lakehouse Files section (relative)."""
        return DEMO_FILES_BASE if self.source_mode == "demo" else LIVE_FILES_BASE

    def table(self, name: str) -> str:
        return f"{self.table_prefix}_{name}"

    @property
    def tables(self) -> dict:
        return {
            "bronze": self.table("bronze_records"),
            "quarantine": self.table("bronze_quarantine"),
            "silver": self.table("silver_theses"),
            "harvest_log": self.table("harvest_log"),
            "gold_by_year": self.table("gold_theses_by_year"),
            "gold_by_university_year": self.table("gold_theses_by_university_year"),
            "gold_completeness": self.table("gold_metadata_completeness"),
        }

    def to_dict(self) -> dict:
        return asdict(self)
''')

In [ ]:
# Embedded module diva_lakehouse.oai_parsing (generated from src/diva_lakehouse/oai_parsing.py)
_diva_register('diva_lakehouse.oai_parsing', r'''"""Pure-Python parsing of OAI-PMH 2.0 responses (Identify, ListRecords + oai_dc).

Only the standard library is used (xml.etree.ElementTree). Namespaces are
handled explicitly; no element is matched by prefix.
"""

from __future__ import annotations

import xml.etree.ElementTree as ET
from dataclasses import dataclass, field
from datetime import datetime, timezone

NS = {
    "oai": "http://www.openarchives.org/OAI/2.0/",
    "oai_dc": "http://www.openarchives.org/OAI/2.0/oai_dc/",
    "dc": "http://purl.org/dc/elements/1.1/",
}

# Dublin Core elements carried into Bronze. dc:creator / dc:contributor are
# deliberately NOT kept (personal names are not needed for this project).
DC_FIELDS = ("title", "subject", "description", "publisher", "date", "type", "identifier", "language")

MAX_SNIPPET_CHARS = 4000


class OAIXMLError(ValueError):
    """The response body is not well-formed XML or not an OAI-PMH document."""


@dataclass
class OAIErrorInfo:
    code: str
    message: str


@dataclass
class ParsedRecord:
    index: int  # 0-based position within the page
    identifier: str | None
    datestamp_raw: str | None
    datestamp: datetime | None
    set_specs: list
    deleted: bool
    dc: dict  # field -> list[str]
    problems: list = field(default_factory=list)
    xml_snippet: str = ""

    @property
    def is_valid(self) -> bool:
        return not self.problems


@dataclass
class ListRecordsPage:
    records: list
    resumption_token: str | None
    complete_list_size: int | None
    cursor: int | None
    errors: list  # list[OAIErrorInfo]
    response_date: str | None

    @property
    def no_records_match(self) -> bool:
        return any(e.code == "noRecordsMatch" for e in self.errors)

    @property
    def fatal_errors(self) -> list:
        return [e for e in self.errors if e.code != "noRecordsMatch"]


def _q(prefix: str, local: str) -> str:
    return f"{{{NS[prefix]}}}{local}"


def _text(el) -> str | None:
    if el is None or el.text is None:
        return None
    value = el.text.strip()
    return value or None


def parse_datestamp(value: str | None) -> datetime | None:
    """Parse an OAI datestamp (day or second granularity, UTC)."""
    if not value:
        return None
    value = value.strip()
    for fmt in ("%Y-%m-%dT%H:%M:%SZ", "%Y-%m-%d"):
        try:
            return datetime.strptime(value, fmt).replace(tzinfo=timezone.utc)
        except ValueError:
            continue
    return None


def _root(xml_bytes: bytes):
    try:
        root = ET.fromstring(xml_bytes)
    except ET.ParseError as exc:
        raise OAIXMLError(f"Response is not well-formed XML: {exc}") from exc
    if root.tag != _q("oai", "OAI-PMH"):
        raise OAIXMLError(f"Unexpected root element {root.tag!r}; expected OAI-PMH 2.0")
    return root


def parse_errors(root) -> list:
    return [
        OAIErrorInfo(code=e.get("code", "unknown"), message=(e.text or "").strip())
        for e in root.findall("oai:error", NS)
    ]


def parse_identify(xml_bytes: bytes) -> dict:
    root = _root(xml_bytes)
    errors = parse_errors(root)
    if errors:
        raise OAIXMLError("Identify returned OAI errors: " + "; ".join(f"{e.code}: {e.message}" for e in errors))
    ident = root.find("oai:Identify", NS)
    if ident is None:
        raise OAIXMLError("Identify element missing in response")
    fields = ("repositoryName", "baseURL", "protocolVersion", "earliestDatestamp", "deletedRecord", "granularity")
    return {f: _text(ident.find(f"oai:{f}", NS)) for f in fields}


def _parse_record(index: int, rec_el) -> ParsedRecord:
    problems = []
    header = rec_el.find("oai:header", NS)
    if header is None:
        return ParsedRecord(index, None, None, None, [], False, {}, ["missing header"], _snippet(rec_el))
    deleted = header.get("status") == "deleted"
    identifier = _text(header.find("oai:identifier", NS))
    datestamp_raw = _text(header.find("oai:datestamp", NS))
    datestamp = parse_datestamp(datestamp_raw)
    set_specs = [s for s in (_text(x) for x in header.findall("oai:setSpec", NS)) if s]
    if not identifier:
        problems.append("missing header identifier")
    if not datestamp_raw:
        problems.append("missing header datestamp")
    elif datestamp is None:
        problems.append(f"invalid header datestamp {datestamp_raw!r}")

    dc_values = {f: [] for f in DC_FIELDS}
    if not deleted:
        dc_el = rec_el.find("oai:metadata/oai_dc:dc", NS)
        if dc_el is None:
            problems.append("missing oai_dc metadata on non-deleted record")
        else:
            for f in DC_FIELDS:
                for el in dc_el.findall(f"dc:{f}", NS):
                    # itertext keeps text of any (unexpected) child markup.
                    raw = "".join(el.itertext())
                    if raw.strip():
                        dc_values[f].append(raw)
    return ParsedRecord(index, identifier, datestamp_raw, datestamp, set_specs, deleted, dc_values, problems,
                        _snippet(rec_el))


def _snippet(el) -> str:
    try:
        text = ET.tostring(el, encoding="unicode")
    except Exception:  # pragma: no cover - defensive
        return ""
    return text[:MAX_SNIPPET_CHARS]


def parse_list_records(xml_bytes: bytes) -> ListRecordsPage:
    root = _root(xml_bytes)
    errors = parse_errors(root)
    response_date = _text(root.find("oai:responseDate", NS))
    lr = root.find("oai:ListRecords", NS)
    if lr is None:
        if not errors:
            raise OAIXMLError("Neither ListRecords nor error element present")
        return ListRecordsPage([], None, None, None, errors, response_date)

    records = [_parse_record(i, r) for i, r in enumerate(lr.findall("oai:record", NS))]
    token_el = lr.find("oai:resumptionToken", NS)
    token, size, cursor = None, None, None
    if token_el is not None:
        token = _text(token_el)  # empty token == last page
        size = _int_or_none(token_el.get("completeListSize"))
        cursor = _int_or_none(token_el.get("cursor"))
    return ListRecordsPage(records, token, size, cursor, errors, response_date)


def _int_or_none(value):
    try:
        return int(value) if value is not None else None
    except ValueError:
        return None
''')

In [ ]:
# Embedded module diva_lakehouse.oai_client (generated from src/diva_lakehouse/oai_client.py)
_diva_register('diva_lakehouse.oai_client', r'''"""Small, polite OAI-PMH 2.0 HTTP client (stdlib only).

* Identify for an endpoint check.
* ListRecords with resumptionToken paging (pages are yielded as raw bytes so
  they can be stored unchanged).
* Timeouts, bounded retries with exponential backoff, Retry-After support,
  a fixed delay between requests.
"""

from __future__ import annotations

import time
import urllib.error
import urllib.parse
import urllib.request
from dataclasses import dataclass

from .oai_parsing import ListRecordsPage, OAIXMLError, parse_identify, parse_list_records

RETRY_STATUSES = {429, 500, 502, 503, 504}
MAX_RETRY_AFTER_S = 120.0


class OAIHTTPError(RuntimeError):
    def __init__(self, status: int, url: str, message: str = ""):
        super().__init__(f"HTTP {status} for {url} {message}".strip())
        self.status = status
        self.url = url


class OAIProtocolError(RuntimeError):
    """The repository answered with an OAI-PMH error other than noRecordsMatch."""

    def __init__(self, errors, url: str):
        text = "; ".join(f"{e.code}: {e.message}" for e in errors)
        super().__init__(f"OAI-PMH error from {url}: {text}")
        self.errors = errors
        self.url = url


@dataclass
class HTTPResponse:
    status: int
    headers: dict
    body: bytes


def urllib_transport(url: str, timeout: float, headers: dict) -> HTTPResponse:
    """Default transport. HTTP error statuses are returned, network errors raised."""
    req = urllib.request.Request(url, headers=headers, method="GET")
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:  # noqa: S310 - URL is validated config
            return HTTPResponse(resp.status, {k.lower(): v for k, v in resp.headers.items()}, resp.read())
    except urllib.error.HTTPError as exc:
        body = exc.read() if hasattr(exc, "read") else b""
        hdrs = {k.lower(): v for k, v in (exc.headers.items() if exc.headers else [])}
        return HTTPResponse(exc.code, hdrs, body)


@dataclass
class FetchedPage:
    page_number: int  # 1-based
    url: str
    request_params: dict
    http_status: int
    body: bytes
    parsed: ListRecordsPage | None
    parse_error: str | None
    attempts: int


class OAIClient:
    def __init__(
        self,
        endpoint: str,
        *,
        transport=urllib_transport,
        timeout_s: float = 60.0,
        max_retries: int = 3,
        backoff_base_s: float = 2.0,
        polite_delay_s: float = 1.0,
        user_agent: str = "diva-research-lakehouse",
        sleep=time.sleep,
        log=print,
    ):
        self.endpoint = endpoint
        self.transport = transport
        self.timeout_s = timeout_s
        self.max_retries = max_retries
        self.backoff_base_s = backoff_base_s
        self.polite_delay_s = polite_delay_s
        self.headers = {"User-Agent": user_agent, "Accept": "text/xml, application/xml"}
        self.sleep = sleep
        self.log = log
        self._requests_made = 0

    # ------------------------------------------------------------------ http
    def build_url(self, params: dict) -> str:
        return self.endpoint + "?" + urllib.parse.urlencode(params)

    def _get(self, params: dict) -> tuple:
        url = self.build_url(params)
        if self._requests_made and self.polite_delay_s:
            self.sleep(self.polite_delay_s)
        attempt = 0
        while True:
            attempt += 1
            self._requests_made += 1
            try:
                resp = self.transport(url, self.timeout_s, self.headers)
            except (urllib.error.URLError, TimeoutError, ConnectionError, OSError) as exc:
                if attempt > self.max_retries:
                    raise OAIHTTPError(0, url, f"network error after {attempt} attempts: {exc}") from exc
                wait = self.backoff_base_s * (2 ** (attempt - 1))
                self.log(f"  network error ({exc}); retry {attempt}/{self.max_retries} in {wait:.1f}s")
                self.sleep(wait)
                continue
            if resp.status == 200:
                return url, resp, attempt
            if resp.status in RETRY_STATUSES and attempt <= self.max_retries:
                wait = self._retry_after(resp) or self.backoff_base_s * (2 ** (attempt - 1))
                self.log(f"  HTTP {resp.status}; retry {attempt}/{self.max_retries} in {wait:.1f}s")
                self.sleep(wait)
                continue
            raise OAIHTTPError(resp.status, url, f"after {attempt} attempt(s)")

    @staticmethod
    def _retry_after(resp: HTTPResponse):
        value = resp.headers.get("retry-after")
        if value is None:
            return None
        try:
            return min(float(value), MAX_RETRY_AFTER_S)
        except ValueError:
            return None

    # ------------------------------------------------------------------ verbs
    def identify(self) -> dict:
        _, resp, _ = self._get({"verb": "Identify"})
        info = parse_identify(resp.body)
        if info.get("protocolVersion") != "2.0":
            raise OAIXMLError(f"Endpoint reports protocolVersion {info.get('protocolVersion')!r}, expected 2.0")
        return info

    def iter_list_records(self, metadata_prefix: str, from_: str | None, until: str | None,
                          set_spec: str | None, max_pages: int):
        """Yield FetchedPage objects until the list ends, an error occurs or max_pages is hit.

        The caller decides when to stop (e.g. at max_records) by breaking out.
        A page whose body is not parseable is yielded with parse_error set and
        iteration stops (there is no reliable resumptionToken to continue with).
        """
        params = {"verb": "ListRecords", "metadataPrefix": metadata_prefix}
        if from_:
            params["from"] = from_
        if until:
            params["until"] = until
        if set_spec:
            params["set"] = set_spec
        page_no = 0
        while page_no < max_pages:
            page_no += 1
            url, resp, attempts = self._get(params)
            try:
                parsed = parse_list_records(resp.body)
                parse_error = None
            except OAIXMLError as exc:
                parsed, parse_error = None, str(exc)
            yield FetchedPage(page_no, url, dict(params), resp.status, resp.body, parsed, parse_error, attempts)
            if parsed is None or parsed.errors or not parsed.resumption_token:
                return
            params = {"verb": "ListRecords", "resumptionToken": parsed.resumption_token}
''')

In [ ]:
# Embedded module diva_lakehouse.selection (generated from src/diva_lakehouse/selection.py)
_diva_register('diva_lakehouse.selection', r'''"""Student-thesis selection based on documented dc:type values.

Rule (documented in docs/data-dictionary.md):
* A record is selected when any dc:type value contains one of
  STUDENT_THESIS_PATTERNS (case-insensitive, whitespace-normalised).
* Research degree theses (doctoral / licentiate) are explicitly NOT student
  theses and are excluded even if a generic pattern would match.
* Deleted headers have no metadata; they are kept so that Silver can mark a
  previously selected record as deleted.

The DiVA dc:type vocabulary could not be verified against the live endpoint
from the build environment; the patterns are configurable in one place.
"""

from __future__ import annotations

import re

STUDENT_THESIS_PATTERNS = (
    "student thesis",
    "studentthesis",
    "studentuppsats",
    "student paper",
    "examensarbete",
    "independent thesis",
    "bachelorthesis",
    "bachelor thesis",
    "masterthesis",
    "master thesis",
    "kandidatuppsats",
    "magisteruppsats",
    "masteruppsats",
)

EXCLUDE_PATTERNS = (
    "doctoral thesis",
    "doctoralthesis",
    "doktorsavhandling",
    "licentiate thesis",
    "licentiatethesis",
    "licentiatavhandling",
)


def _norm(value: str) -> str:
    return re.sub(r"\s+", " ", value).strip().casefold()


def classify_record(types: list, deleted: bool, thesis_filter: str = "student_thesis") -> tuple:
    """Return (is_selected, reason)."""
    if deleted:
        return True, "deleted header (no metadata); kept to propagate deletion"
    if thesis_filter == "none":
        return True, "thesis_filter=none (all records selected)"
    normed = [_norm(t) for t in types or []]
    for t in normed:
        for pat in EXCLUDE_PATTERNS:
            if pat in t:
                return False, f"dc:type matched exclusion {pat!r}"
    for t in normed:
        for pat in STUDENT_THESIS_PATTERNS:
            if pat in t:
                return True, f"dc:type matched {pat!r}"
    if not normed:
        return False, "no dc:type value"
    return False, "no dc:type value matched student-thesis patterns"
''')

In [ ]:
# Embedded module diva_lakehouse.cleaning (generated from src/diva_lakehouse/cleaning.py)
_diva_register('diva_lakehouse.cleaning', r'''"""Pure-Python cleaning rules for Silver (testable without Spark).

All rules are documented in docs/data-dictionary.md. Original values are kept
next to every derived value.
"""

from __future__ import annotations

import hashlib
import json
import re
import unicodedata
from datetime import datetime, timezone

# --------------------------------------------------------------------- text
_WS_INLINE = re.compile(r"[ \t\f\v  -​  　]+")
_WS_ANY = re.compile(r"\s+")


def normalize_text(value: str | None) -> str | None:
    """NFC + collapse all whitespace (incl. newlines) to one space + trim."""
    if value is None:
        return None
    value = _WS_ANY.sub(" ", unicodedata.normalize("NFC", value)).strip()
    return value or None


def normalize_abstract(value: str | None) -> str | None:
    """Whitespace-only normalisation that keeps paragraph breaks.

    Words, punctuation and order are never changed: line endings become \\n,
    runs of spaces/tabs become one space, lines are trimmed and 3+ consecutive
    line breaks become one blank line.
    """
    if value is None:
        return None
    value = unicodedata.normalize("NFC", value).replace("\r\n", "\n").replace("\r", "\n")
    lines = [_WS_INLINE.sub(" ", line).strip() for line in value.split("\n")]
    value = re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()
    return value or None


def clean_list(values) -> list:
    """Normalise each value, drop empties, de-duplicate keeping first occurrence."""
    out, seen = [], set()
    for v in values or []:
        n = normalize_text(v)
        if n and n not in seen:
            seen.add(n)
            out.append(n)
    return out


# ----------------------------------------------------------------- language
LANGUAGE_MAP = {
    "sv": "sv", "swe": "sv", "sve": "sv", "sv-se": "sv", "svenska": "sv", "swedish": "sv",
    "en": "en", "eng": "en", "en-gb": "en", "en-us": "en", "engelska": "en", "english": "en",
    "no": "no", "nor": "no", "nb": "no", "nob": "no", "nn": "no", "nno": "no", "norska": "no",
    "norwegian": "no",
    "da": "da", "dan": "da", "danska": "da", "danish": "da",
    "fi": "fi", "fin": "fi", "finska": "fi", "finnish": "fi",
    "de": "de", "ger": "de", "deu": "de", "tyska": "de", "german": "de",
    "fr": "fr", "fre": "fr", "fra": "fr", "franska": "fr", "french": "fr",
    "es": "es", "spa": "es", "spanska": "es", "spanish": "es",
    "it": "it", "ita": "it", "italienska": "it", "italian": "it",
}
UNKNOWN_LANGUAGE = "und"


def normalize_language(values) -> tuple:
    """Return (primary_code, all_codes). Unmapped values -> 'und'."""
    codes = []
    for v in values or []:
        key = (normalize_text(v) or "").casefold()
        code = LANGUAGE_MAP.get(key, UNKNOWN_LANGUAGE)
        if code not in codes:
            codes.append(code)
    mapped = [c for c in codes if c != UNKNOWN_LANGUAGE]
    primary = mapped[0] if mapped else UNKNOWN_LANGUAGE
    return primary, codes


# ---------------------------------------------------------- publication year
_YEAR_RE = re.compile(r"^(\d{4})(?:-(\d{2})(?:-(\d{2}))?)?(?:T[\d:.]+Z?)?$")
MIN_YEAR = 1900


def parse_publication_year(values, max_year: int | None = None) -> tuple:
    """Return (year | None, status). status in parsed|missing|unparseable|ambiguous.

    Cautious rule: only values that are a plain ISO-like date beginning with a
    four-digit year are accepted, the year must be within [1900, max_year],
    and if the accepted values disagree on the year the result is ambiguous.
    """
    max_year = max_year or datetime.now(timezone.utc).year + 1
    cleaned = [normalize_text(v) for v in values or []]
    cleaned = [v for v in cleaned if v]
    if not cleaned:
        return None, "missing"
    years = set()
    for v in cleaned:
        m = _YEAR_RE.match(v)
        if m and MIN_YEAR <= int(m.group(1)) <= max_year:
            years.add(int(m.group(1)))
    if not years:
        return None, "unparseable"
    if len(years) > 1:
        return None, "ambiguous"
    return years.pop(), "parsed"


# --------------------------------------------------------------- university
# Heuristic, documented mapping. dc:publisher in DiVA is free text and may name
# a department, a university or a publisher, so it is NOT assumed to be a
# university. A record is attributed to a university only when the text
# matches one of these name patterns (word-bounded, case-insensitive).
UNIVERSITY_PATTERNS = {
    "Uppsala universitet": ("uppsala universitet", "uppsala university"),
    "KTH Kungliga Tekniska högskolan": ("kth", "kungliga tekniska högskolan", "royal institute of technology"),
    "Stockholms universitet": ("stockholms universitet", "stockholm university"),
    "Linköpings universitet": ("linköpings universitet", "linköping university"),
    "Umeå universitet": ("umeå universitet", "umeå university"),
    "Örebro universitet": ("örebro universitet", "örebro university"),
    "Luleå tekniska universitet": ("luleå tekniska universitet", "luleå university of technology"),
    "Mittuniversitetet": ("mittuniversitetet", "mid sweden university"),
    "Karlstads universitet": ("karlstads universitet", "karlstad university"),
    "Linnéuniversitetet": ("linnéuniversitetet", "linnaeus university"),
    "Mälardalens universitet": ("mälardalens universitet", "mälardalens högskola", "mälardalen university"),
    "Malmö universitet": ("malmö universitet", "malmö högskola", "malmö university"),
    "Södertörns högskola": ("södertörns högskola", "södertörn university"),
    "Högskolan i Gävle": ("högskolan i gävle", "university of gävle"),
    "Högskolan Dalarna": ("högskolan dalarna", "dalarna university"),
    "Jönköping University": ("jönköping university", "högskolan i jönköping"),
    "Blekinge Tekniska Högskola": ("blekinge tekniska högskola", "blekinge institute of technology"),
    "Högskolan i Skövde": ("högskolan i skövde", "university of skövde"),
    "Högskolan i Borås": ("högskolan i borås", "university of borås"),
    "Högskolan i Halmstad": ("högskolan i halmstad", "halmstad university"),
    "Högskolan Kristianstad": ("högskolan kristianstad", "kristianstad university"),
    "Högskolan Väst": ("högskolan väst", "university west"),
    "Gymnastik- och idrottshögskolan": ("gymnastik- och idrottshögskolan", "swedish school of sport and health sciences"),
    "Försvarshögskolan": ("försvarshögskolan", "swedish defence university"),
}

# Secondary heuristic: DiVA OAI identifiers are commonly of the form
# "oai:DiVA.org:<code>-<number>". The code is only used when no publisher
# matched. (Not verified against the live endpoint from the build env.)
IDENTIFIER_CODE_MAP = {
    "uu": "Uppsala universitet", "kth": "KTH Kungliga Tekniska högskolan", "su": "Stockholms universitet",
    "liu": "Linköpings universitet", "umu": "Umeå universitet", "oru": "Örebro universitet",
    "ltu": "Luleå tekniska universitet", "miun": "Mittuniversitetet", "kau": "Karlstads universitet",
    "lnu": "Linnéuniversitetet", "mdh": "Mälardalens universitet", "mau": "Malmö universitet",
    "sh": "Södertörns högskola", "hig": "Högskolan i Gävle", "du": "Högskolan Dalarna",
    "hj": "Jönköping University", "bth": "Blekinge Tekniska Högskola", "his": "Högskolan i Skövde",
    "hb": "Högskolan i Borås", "hh": "Högskolan i Halmstad", "hkr": "Högskolan Kristianstad",
    "hv": "Högskolan Väst", "gih": "Gymnastik- och idrottshögskolan", "fhs": "Försvarshögskolan",
}
_DIVA_ID_RE = re.compile(r"^oai:diva\.org:([a-z]+)-\d+$", re.IGNORECASE)

UNKNOWN_UNIVERSITY = "Okänt lärosäte"
MULTIPLE_UNIVERSITIES = "Flera lärosäten (ej entydigt)"

_COMPILED_UNI = {
    name: [re.compile(r"(?<!\w)" + re.escape(p) + r"(?!\w)") for p in pats]
    for name, pats in UNIVERSITY_PATTERNS.items()
}


def derive_university(publishers, record_id: str | None) -> dict:
    """Return dict(university, university_group, method, candidates).

    university        : canonical name when exactly one university is found, else None
    university_group  : value used by Gold (never double counts a record):
                        the university, MULTIPLE_UNIVERSITIES or UNKNOWN_UNIVERSITY
    method            : publisher_match | identifier_prefix | multiple_publisher_matches | unknown
    candidates        : all universities matched in dc:publisher (sorted)
    """
    found = set()
    for p in publishers or []:
        text = (normalize_text(p) or "").casefold()
        for name, regexes in _COMPILED_UNI.items():
            if any(r.search(text) for r in regexes):
                found.add(name)
    candidates = sorted(found)
    if len(candidates) == 1:
        return {"university": candidates[0], "university_group": candidates[0],
                "method": "publisher_match", "candidates": candidates}
    if len(candidates) > 1:
        return {"university": None, "university_group": MULTIPLE_UNIVERSITIES,
                "method": "multiple_publisher_matches", "candidates": candidates}
    m = _DIVA_ID_RE.match(record_id or "")
    if m and m.group(1).lower() in IDENTIFIER_CODE_MAP:
        name = IDENTIFIER_CODE_MAP[m.group(1).lower()]
        return {"university": name, "university_group": name, "method": "identifier_prefix", "candidates": []}
    return {"university": None, "university_group": UNKNOWN_UNIVERSITY, "method": "unknown", "candidates": []}


# ----------------------------------------------------------------- links/hash
def pick_source_url(identifiers) -> str | None:
    urls = [i for i in (normalize_text(x) for x in identifiers or []) if i and i.lower().startswith(("http://", "https://"))]
    if not urls:
        return None
    diva = [u for u in urls if "diva-portal.org" in u.lower()]
    return (diva or urls)[0]


HASH_FIELDS = ("record_id", "title", "titles", "abstract", "abstracts", "subjects", "date_raw", "language_raw",
               "publishers", "document_types", "dc_identifiers")
_ORDER_INSENSITIVE = {"subjects", "publishers", "document_types", "dc_identifiers"}


def content_hash(row: dict) -> str:
    """SHA-256 over the business fields only (no run_id, no timestamps).

    Lists whose order carries no meaning are sorted, so a reordering in the
    source does not count as a change.
    """
    payload = {}
    for f in HASH_FIELDS:
        v = row.get(f)
        if isinstance(v, (list, tuple)):
            v = sorted(v) if f in _ORDER_INSENSITIVE else list(v)
        payload[f] = v
    blob = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()


def build_silver_candidate(obs: dict) -> dict:
    """Map one Bronze observation (dict) to Silver business fields + hash."""
    record_id = obs["record_id"]
    titles = clean_list(obs.get("dc_title"))
    abstracts = []
    for a in obs.get("dc_description") or []:
        n = normalize_abstract(a)
        if n and n not in abstracts:
            abstracts.append(n)
    publishers = clean_list(obs.get("dc_publisher"))
    date_raw = [d for d in (obs.get("dc_date") or []) if d is not None]
    language_raw = [x for x in (obs.get("dc_language") or []) if x is not None]
    lang, lang_codes = normalize_language(language_raw)
    year, year_status = parse_publication_year(date_raw)
    uni = derive_university(publishers, record_id)
    row = {
        "record_id": record_id,
        "title": titles[0] if titles else None,
        "titles": titles,
        "abstract": abstracts[0] if abstracts else None,
        "abstracts": abstracts,
        "subjects": clean_list(obs.get("dc_subject")),
        "publication_year": year,
        "publication_year_status": year_status,
        "date_raw": date_raw,
        "language": lang,
        "language_codes": lang_codes,
        "language_raw": language_raw,
        "publishers": publishers,
        "university": uni["university"],
        "university_group": uni["university_group"],
        "university_method": uni["method"],
        "university_candidates": uni["candidates"],
        "document_types": clean_list(obs.get("dc_type")),
        "dc_identifiers": clean_list(obs.get("dc_identifier")),
        "source_url": pick_source_url(obs.get("dc_identifier")),
        "set_specs": list(obs.get("set_specs") or []),
    }
    row["content_hash"] = content_hash(row)
    return row
''')

In [ ]:
# Embedded module diva_lakehouse.storage (generated from src/diva_lakehouse/storage.py)
_diva_register('diva_lakehouse.storage', r'''"""Lakehouse Files access through the default-Lakehouse file mount.

In a Fabric notebook with a default Lakehouse attached, the Lakehouse is
mounted at /lakehouse/default/ (documented in "Load data into your lakehouse
with a notebook"). Files are written with plain Python I/O under
/lakehouse/default/Files/...; Delta tables are created with saveAsTable in the
default Lakehouse. No workspace ID or ABFS path is needed.

Locally (tests, scripts/run_local_demo.py) any directory can act as the
Lakehouse root.
"""

from __future__ import annotations

import hashlib
import json
import os
import shutil

FABRIC_DEFAULT_LAKEHOUSE = "/lakehouse/default"

MISSING_LAKEHOUSE_HELP = (
    "No default Lakehouse is available to this notebook ({root} does not exist or has no Files folder).\n"
    "Fix: in the notebook, open the 'Explorer' / 'Lakehouses' pane, choose 'Add' (or 'Add data items'), "
    "select an existing Lakehouse or create a new one, and make sure it is pinned as the default Lakehouse. "
    "Then restart the session (Stop session) and run the notebook again. "
    "See docs/setup-fabric.md, step 5."
)


class LakehouseNotAvailable(RuntimeError):
    pass


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


class Lakehouse:
    """Access to the Files section of a Lakehouse via a local (mounted) path."""

    def __init__(self, root: str = "auto"):
        self.root = FABRIC_DEFAULT_LAKEHOUSE if root in ("", "auto", None) else root

    @property
    def files_dir(self) -> str:
        return os.path.join(self.root, "Files")

    def check_available(self) -> None:
        """Fail early with an actionable message if no Lakehouse is attached."""
        if not os.path.isdir(self.files_dir):
            raise LakehouseNotAvailable(MISSING_LAKEHOUSE_HELP.format(root=self.root))
        probe = os.path.join(self.files_dir, ".diva_write_probe")
        try:
            with open(probe, "w", encoding="utf-8") as fh:
                fh.write("ok")
            os.remove(probe)
        except OSError as exc:
            raise LakehouseNotAvailable(
                f"The default Lakehouse Files folder is not writable: {exc}\n" + MISSING_LAKEHOUSE_HELP.format(root=self.root)
            ) from exc

    # ---------------------------------------------------------------- paths
    def path(self, rel: str) -> str:
        """Local path for a path relative to the Files section ('diva/raw/...')."""
        rel = rel.lstrip("/")
        if rel.startswith("Files/"):
            rel = rel[len("Files/"):]
        return os.path.join(self.files_dir, rel)

    @staticmethod
    def display(rel: str) -> str:
        """Lakehouse-relative path as shown in Fabric ('Files/diva/...')."""
        rel = rel.lstrip("/")
        return rel if rel.startswith("Files/") else "Files/" + rel

    def exists(self, rel: str) -> bool:
        return os.path.exists(self.path(rel))

    # ------------------------------------------------------------------- io
    def write_bytes(self, rel: str, data: bytes) -> str:
        p = self.path(rel)
        os.makedirs(os.path.dirname(p), exist_ok=True)
        tmp = p + ".tmp"
        with open(tmp, "wb") as fh:
            fh.write(data)
        os.replace(tmp, p)
        return p

    def read_bytes(self, rel: str) -> bytes:
        with open(self.path(rel), "rb") as fh:
            return fh.read()

    def write_json(self, rel: str, obj) -> str:
        data = json.dumps(obj, ensure_ascii=False, indent=2, sort_keys=False, default=str).encode("utf-8")
        return self.write_bytes(rel, data)

    def read_json(self, rel: str):
        return json.loads(self.read_bytes(rel).decode("utf-8"))

    def write_text(self, rel: str, text: str) -> str:
        return self.write_bytes(rel, text.encode("utf-8"))

    def read_text(self, rel: str) -> str:
        return self.read_bytes(rel).decode("utf-8")

    def listdir(self, rel: str) -> list:
        p = self.path(rel)
        return sorted(os.listdir(p)) if os.path.isdir(p) else []

    def remove_tree(self, rel: str) -> None:
        p = self.path(rel)
        if os.path.isdir(p):
            shutil.rmtree(p)
''')

In [ ]:
# Embedded module diva_lakehouse.demo_data (generated from src/diva_lakehouse/demo_data.py)
_diva_register('diva_lakehouse.demo_data', r'''"""SYNTHETIC OAI-PMH fixtures for demo mode and tests.

Every record here is invented. Titles start with "[SYNTETISK]" and identifiers
use the reserved example domain "synthetic.example". Results produced from
these fixtures are demo output, not statistics about DiVA.

Batch A (two pages, linked with a resumptionToken):
  demo-0001  new student thesis (Uppsala, Swedish, 2 subjects)
  demo-0002  new student thesis (KTH, English)
  demo-0090  journal article -> filtered out by the student-thesis rule
  demo-0003  new student thesis with publishers from two universities
  demo-0004  new student thesis without publisher/abstract
  (no id)    broken record without identifier -> quarantine
Batch B (one page):
  demo-0002  changed (revised abstract, newer datestamp)
  demo-0005  new
  demo-0003  unchanged content, newer datestamp
  demo-0004  deleted header
"""

from __future__ import annotations

import urllib.parse

from .oai_client import HTTPResponse

SYNTHETIC_ENDPOINT = "synthetic://demo-oai"

_HEAD = """<?xml version="1.0" encoding="UTF-8"?>
<!-- SYNTHETIC DEMO DATA - not real DiVA records -->
<OAI-PMH xmlns="http://www.openarchives.org/OAI/2.0/"
         xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <responseDate>{response_date}</responseDate>
  <request verb="ListRecords" metadataPrefix="oai_dc">synthetic://demo-oai</request>
"""
_TAIL = "</OAI-PMH>\n"

_DC_OPEN = ('<oai_dc:dc xmlns:oai_dc="http://www.openarchives.org/OAI/2.0/oai_dc/" '
            'xmlns:dc="http://purl.org/dc/elements/1.1/">')


def _record(identifier, datestamp, dc_xml, set_spec="synthetic:studentThesis"):
    ident = f"<identifier>{identifier}</identifier>" if identifier else ""
    return f"""  <record>
    <header>
      {ident}
      <datestamp>{datestamp}</datestamp>
      <setSpec>{set_spec}</setSpec>
    </header>
    <metadata>
      {_DC_OPEN}
{dc_xml}
      </oai_dc:dc>
    </metadata>
  </record>
"""


def _deleted(identifier, datestamp):
    return f"""  <record>
    <header status="deleted">
      <identifier>{identifier}</identifier>
      <datestamp>{datestamp}</datestamp>
    </header>
  </record>
"""


_STUDENT_TYPES = """        <dc:type>Student thesis</dc:type>
        <dc:type>info:eu-repo/semantics/bachelorThesis</dc:type>
        <dc:type>text</dc:type>"""

_R1 = _record("oai:synthetic.example:demo-0001", "2024-03-01T08:15:00Z", f"""        <dc:title>[SYNTETISK] Öppna data i kommunal planering</dc:title>
        <dc:title>[SYNTHETIC] Open data in municipal planning</dc:title>
        <dc:subject>öppna data</dc:subject>
        <dc:subject>kommunal planering</dc:subject>
        <dc:subject>öppna data</dc:subject>
        <dc:description>Syntetisk sammanfattning.   Den här uppsatsen finns inte på riktigt.</dc:description>
        <dc:publisher>Uppsala universitet, Institutionen för informatik och media</dc:publisher>
        <dc:date>2021</dc:date>
{_STUDENT_TYPES}
        <dc:identifier>https://synthetic.example/record/demo-0001</dc:identifier>
        <dc:language>swe</dc:language>""")

_R2_A = _record("oai:synthetic.example:demo-0002", "2024-03-01T09:00:00Z", f"""        <dc:title>[SYNTHETIC] Energy use in student housing</dc:title>
        <dc:subject>energy</dc:subject>
        <dc:description>Synthetic abstract, version 1.</dc:description>
        <dc:publisher>KTH, School of Architecture and the Built Environment (ABE)</dc:publisher>
        <dc:date>2022-06-10</dc:date>
{_STUDENT_TYPES}
        <dc:identifier>https://synthetic.example/record/demo-0002</dc:identifier>
        <dc:language>eng</dc:language>""")

_R90 = _record("oai:synthetic.example:demo-0090", "2024-03-01T09:30:00Z", """        <dc:title>[SYNTHETIC] A journal article that is not a student thesis</dc:title>
        <dc:publisher>Uppsala universitet</dc:publisher>
        <dc:date>2023</dc:date>
        <dc:type>Article in journal</dc:type>
        <dc:type>info:eu-repo/semantics/article</dc:type>
        <dc:language>eng</dc:language>""", set_spec="synthetic:article")

_R3 = _record("oai:synthetic.example:demo-0003", "2024-03-01T10:00:00Z", f"""        <dc:title>[SYNTETISK] Samverkan mellan två lärosäten</dc:title>
        <dc:subject>samverkan</dc:subject>
        <dc:subject>högre utbildning</dc:subject>
        <dc:description>Syntetisk sammanfattning om samverkan.</dc:description>
        <dc:publisher>Stockholms universitet</dc:publisher>
        <dc:publisher>Södertörns högskola</dc:publisher>
        <dc:date>2022-06-01</dc:date>
{_STUDENT_TYPES}
        <dc:language>sv</dc:language>""")

_R4 = _record("oai:synthetic.example:demo-0004", "2024-03-01T11:00:00Z", f"""        <dc:title>[SYNTETISK] Uppsats utan abstract och utgivare</dc:title>
        <dc:date>2023-01-15</dc:date>
{_STUDENT_TYPES}
        <dc:language>swe</dc:language>""")

_BROKEN = _record(None, "2024-03-01T11:30:00Z", f"""        <dc:title>[SYNTETISK] Trasig post utan identifierare</dc:title>
{_STUDENT_TYPES}""")

_R2_B = _record("oai:synthetic.example:demo-0002", "2024-03-05T12:00:00Z", f"""        <dc:title>[SYNTHETIC] Energy use in student housing</dc:title>
        <dc:subject>energy</dc:subject>
        <dc:description>Synthetic abstract, version 2 (revised by the author).</dc:description>
        <dc:publisher>KTH, School of Architecture and the Built Environment (ABE)</dc:publisher>
        <dc:date>2022-06-10</dc:date>
{_STUDENT_TYPES}
        <dc:identifier>https://synthetic.example/record/demo-0002</dc:identifier>
        <dc:language>eng</dc:language>""")

_R3_B = _R3.replace("2024-03-01T10:00:00Z", "2024-03-05T12:10:00Z")

_R5 = _record("oai:synthetic.example:demo-0005", "2024-03-05T12:20:00Z", f"""        <dc:title>[SYNTETISK] Digital tillgänglighet på bibliotek</dc:title>
        <dc:subject>tillgänglighet</dc:subject>
        <dc:subject>bibliotek</dc:subject>
        <dc:description>Syntetisk sammanfattning.

Andra stycket i abstract.</dc:description>
        <dc:publisher>Linköpings universitet, Institutionen för kultur och samhälle</dc:publisher>
        <dc:date>2024</dc:date>
{_STUDENT_TYPES}
        <dc:language>swe</dc:language>""")

_R4_DEL = _deleted("oai:synthetic.example:demo-0004", "2024-03-05T12:30:00Z")


def _page(response_date, records, token_xml):
    return (_HEAD.format(response_date=response_date) + "  <ListRecords>\n" + "".join(records)
            + token_xml + "  </ListRecords>\n" + _TAIL).encode("utf-8")


BATCH_A_PAGES = [
    _page("2024-03-02T00:00:00Z", [_R1, _R2_A, _R90],
          '    <resumptionToken completeListSize="6" cursor="0">demoA-page2</resumptionToken>\n'),
    _page("2024-03-02T00:00:01Z", [_R3, _R4, _BROKEN],
          '    <resumptionToken completeListSize="6" cursor="3"/>\n'),
]
BATCH_B_PAGES = [
    _page("2024-03-06T00:00:00Z", [_R2_B, _R5, _R3_B, _R4_DEL], ""),
]
BATCHES = {"A": BATCH_A_PAGES, "B": BATCH_B_PAGES}

IDENTIFY_XML = b"""<?xml version="1.0" encoding="UTF-8"?>
<OAI-PMH xmlns="http://www.openarchives.org/OAI/2.0/">
  <responseDate>2024-03-01T00:00:00Z</responseDate>
  <request verb="Identify">synthetic://demo-oai</request>
  <Identify>
    <repositoryName>SYNTHETIC demo repository (not DiVA)</repositoryName>
    <baseURL>synthetic://demo-oai</baseURL>
    <protocolVersion>2.0</protocolVersion>
    <adminEmail>nobody@synthetic.example</adminEmail>
    <earliestDatestamp>2024-01-01T00:00:00Z</earliestDatestamp>
    <deletedRecord>persistent</deletedRecord>
    <granularity>YYYY-MM-DDThh:mm:ssZ</granularity>
  </Identify>
</OAI-PMH>
"""

# Expected outcome (used by tests and the re-run demo):
EXPECTED_AFTER_A = {"silver_rows": 4, "silver_active": 4, "quarantine": 1, "not_selected": 1}
EXPECTED_AFTER_B = {"silver_rows": 5, "silver_active": 4, "new": 1, "changed": 1, "unchanged": 1, "deleted": 1}


class SyntheticTransport:
    """Serves the synthetic pages through the same client code as live mode."""

    def __init__(self, batch: str):
        self.pages = BATCHES[batch]
        self.calls = []

    def __call__(self, url, timeout, headers):
        self.calls.append(url)
        query = urllib.parse.parse_qs(urllib.parse.urlsplit(url).query)
        verb = query.get("verb", [""])[0]
        if verb == "Identify":
            return HTTPResponse(200, {}, IDENTIFY_XML)
        if verb == "ListRecords":
            token = query.get("resumptionToken", [None])[0]
            if token is None:
                return HTTPResponse(200, {}, self.pages[0])
            if token == "demoA-page2" and len(self.pages) > 1:
                return HTTPResponse(200, {}, self.pages[1])
        return HTTPResponse(400, {}, b"unsupported synthetic request")
''')

In [ ]:
# Embedded module diva_lakehouse.ingestion (generated from src/diva_lakehouse/ingestion.py)
_diva_register('diva_lakehouse.ingestion', r'''"""01_Ingestion: harvest OAI-PMH pages, store them unchanged, write a manifest.

Raw pages are stored byte-for-byte under <files_base>/raw/<run_id>/ and the
manifest records SHA-256 per file, the request parameters and whether the
harvest is complete. A harvest stopped at max_records / max_pages is marked
complete=False and must never be treated as a finished interval.
"""

from __future__ import annotations

from collections import Counter

from .config import PipelineConfig, iso_utc, utc_now
from .oai_client import OAIClient, OAIProtocolError
from .selection import classify_record
from .storage import Lakehouse, sha256_bytes

MANIFEST_NAME = "manifest.json"


def raw_dir(cfg: PipelineConfig, run_id: str) -> str:
    return f"{cfg.files_base}/raw/{run_id}"


def manifest_path(cfg: PipelineConfig, run_id: str) -> str:
    return f"{raw_dir(cfg, run_id)}/{MANIFEST_NAME}"


def harvest(cfg: PipelineConfig, lakehouse: Lakehouse, client: OAIClient, run_id: str,
            from_datestamp: str | None, until_datestamp: str | None, log=print) -> dict:
    started = utc_now()
    log(f"Identify {client.endpoint} ...")
    identify = client.identify()
    log(f"  repository={identify.get('repositoryName')!r} granularity={identify.get('granularity')} "
        f"deletedRecord={identify.get('deletedRecord')} earliestDatestamp={identify.get('earliestDatestamp')}")
    if from_datestamp and "T" in from_datestamp and identify.get("granularity") == "YYYY-MM-DD":
        raise ValueError("The repository only supports day granularity; use YYYY-MM-DD datestamps.")

    requested = {
        "metadataPrefix": cfg.metadata_prefix,
        "from": from_datestamp,
        "until": until_datestamp,
        "set": cfg.set_spec or None,
        "max_records": cfg.max_records,
        "max_pages": cfg.max_pages,
        "thesis_filter": cfg.thesis_filter,
    }
    pages_meta = []
    used_total = 0
    in_raw_total = 0
    deleted = selected = not_selected = invalid = 0
    type_counter = Counter()
    reason_counter = Counter()
    complete = False
    stop_reason = None
    last_token = None
    oai_error = None

    for page in client.iter_list_records(cfg.metadata_prefix, from_datestamp, until_datestamp,
                                         cfg.set_spec or None, cfg.max_pages):
        fname = f"page_{page.page_number:05d}.xml"
        rel = f"{raw_dir(cfg, run_id)}/{fname}"
        lakehouse.write_bytes(rel, page.body)
        meta = {
            "file": fname,
            "path": lakehouse.display(rel),
            "sha256": sha256_bytes(page.body),
            "fetched_at_utc": iso_utc(utc_now()),
            "bytes": len(page.body),
            "url": page.url,
            "request_params": page.request_params,
            "http_status": page.http_status,
            "attempts": page.attempts,
            "parse_error": page.parse_error,
            "records_in_page": 0,
            "records_used": 0,
            "resumption_token": None,
            "complete_list_size": None,
            "oai_errors": [],
        }
        pages_meta.append(meta)
        if page.parsed is None:
            stop_reason = "page_parse_error"
            log(f"  page {page.page_number}: NOT parseable ({page.parse_error}); stopping (incomplete).")
            break
        parsed = page.parsed
        meta["oai_errors"] = [{"code": e.code, "message": e.message} for e in parsed.errors]
        meta["resumption_token"] = parsed.resumption_token
        meta["complete_list_size"] = parsed.complete_list_size
        if parsed.no_records_match:
            complete, stop_reason = True, "noRecordsMatch"
            log("  noRecordsMatch: the interval/set contains no records (complete, 0 records).")
            break
        if parsed.fatal_errors:
            stop_reason = "oai_error"
            oai_error = OAIProtocolError(parsed.fatal_errors, page.url)
            break

        n = len(parsed.records)
        remaining = cfg.max_records - used_total
        use = min(n, remaining)
        meta["records_in_page"] = n
        meta["records_used"] = use
        in_raw_total += n
        used_total += use
        for rec in parsed.records[:use]:
            if not rec.is_valid:
                invalid += 1
                continue
            if rec.deleted:
                deleted += 1
            for t in rec.dc.get("type", []):
                type_counter[" ".join(t.split())] += 1
            is_sel, reason = classify_record(rec.dc.get("type", []), rec.deleted, cfg.thesis_filter)
            reason_counter[reason] += 1
            if is_sel and not rec.deleted:
                selected += 1
            elif not is_sel:
                not_selected += 1
        log(f"  page {page.page_number}: {n} records (used {use}); total used {used_total}"
            + (f"; completeListSize={parsed.complete_list_size}" if parsed.complete_list_size else ""))
        last_token = parsed.resumption_token
        if used_total >= cfg.max_records and (use < n or last_token):
            stop_reason = "max_records"
            break
        if not last_token:
            complete, stop_reason = True, "list_exhausted"
            break
    else:
        # The page generator ended without a break: max_pages was reached
        # while a resumptionToken was still outstanding (or no page at all).
        stop_reason = "max_pages" if pages_meta else "no_pages"
        complete = False

    manifest = {
        "run_id": run_id,
        "source_mode": cfg.source_mode,
        "endpoint": client.endpoint,
        "identify": identify,
        "requested": requested,
        "fetch_started_at_utc": iso_utc(started),
        "fetch_finished_at_utc": iso_utc(utc_now()),
        "pages": pages_meta,
        "counts": {
            "pages": len(pages_meta),
            "records_in_raw_pages": in_raw_total,
            "records_fetched": used_total,  # records within max_records that are processed
            "records_beyond_max_records": in_raw_total - used_total,
            "selected_theses": selected,
            "not_selected": not_selected,
            "deleted_headers": deleted,
            "invalid_records": invalid,
        },
        "complete": complete,
        "stop_reason": stop_reason,
        "selection": {
            "rule": "dc:type pattern match (see selection.py / docs/data-dictionary.md)",
            "reasons": dict(reason_counter),
            "dc_type_values": dict(type_counter.most_common(50)),
        },
        "notes": [],
    }
    if cfg.source_mode == "demo":
        manifest["notes"].append("SYNTHETIC DEMO DATA - not real DiVA records.")
    if not complete:
        manifest["notes"].append(
            f"Harvest INCOMPLETE (stop_reason={stop_reason}). The interval is not registered as completed; "
            "narrow the datestamp interval or raise max_records and run again.")
    if cfg.source_mode == "live" and selected < cfg.low_selection_threshold and stop_reason != "oai_error":
        manifest["notes"].append(
            f"Few student theses selected ({selected}). Possible reasons: the datestamp interval is short; "
            "most changed records in the interval are not student theses; no set_spec restricts the harvest to "
            "theses; or the dc:type vocabulary differs from the documented patterns (see 'dc_type_values'). "
            "No data is fabricated to compensate.")
    lakehouse.write_json(manifest_path(cfg, run_id), manifest)
    for note in manifest["notes"]:
        log("  NOTE: " + note)
    if oai_error is not None:
        raise oai_error
    return manifest
''')

In [ ]:
# Embedded module diva_lakehouse.spark_utils (generated from src/diva_lakehouse/spark_utils.py)
_diva_register('diva_lakehouse.spark_utils', r'''"""Spark / Delta helpers shared by the Bronze, Silver and Gold stages.

In Fabric the notebook already provides a `spark` session with Delta Lake
configured; get_spark() just returns it. Locally a Delta-enabled session is
created with delta-spark (tests and scripts/run_local_demo.py only).
"""

from __future__ import annotations


def get_spark(local_warehouse: str | None = None, app_name: str = "diva-lakehouse"):
    from pyspark.sql import SparkSession

    active = SparkSession.getActiveSession()
    if active is not None:
        return active
    if local_warehouse is None:
        return SparkSession.builder.getOrCreate()
    import os
    import sys

    from delta import configure_spark_with_delta_pip  # local only

    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

    builder = (
        SparkSession.builder.master("local[2]")
        .appName(app_name)
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.warehouse.dir", local_warehouse)
        .config("spark.sql.shuffle.partitions", "2")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.ui.enabled", "false")
        .config("spark.driver.extraJavaOptions", f"-Dderby.system.home={local_warehouse}/_derby")
    )
    return configure_spark_with_delta_pip(builder).getOrCreate()


def table_exists(spark, name: str) -> bool:
    return bool(spark.catalog.tableExists(name))


def ensure_table(spark, name: str, schema) -> None:
    """Create an empty Delta table with an explicit schema if it does not exist."""
    if not table_exists(spark, name):
        spark.createDataFrame([], schema).write.format("delta").saveAsTable(name)


def merge_insert_only(spark, name: str, df, key: str) -> int:
    """Insert rows whose key is not yet present (idempotent append).

    Returns the number of inserted rows (row count after - before), which is
    also correct when the MERGE inserts nothing.
    """
    from delta.tables import DeltaTable

    ensure_table(spark, name, df.schema)
    before = spark.table(name).count()
    (
        DeltaTable.forName(spark, name).alias("t")
        .merge(df.alias("s"), f"t.{key} = s.{key}")
        .whenNotMatchedInsertAll()
        .execute()
    )
    return spark.table(name).count() - before


def last_operation_metrics(spark, name: str) -> dict:
    from delta.tables import DeltaTable

    row = DeltaTable.forName(spark, name).history(1).select("operation", "operationMetrics", "version").first()
    metrics = dict(row["operationMetrics"] or {})
    return {"operation": row["operation"], "version": row["version"],
            **{k: int(v) for k, v in metrics.items() if str(v).isdigit()}}


def overwrite_table(df, name: str) -> None:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)


def drop_table(spark, name: str) -> None:
    spark.sql(f"DROP TABLE IF EXISTS {name}")
''')

In [ ]:
# Embedded module diva_lakehouse.bronze (generated from src/diva_lakehouse/bronze.py)
_diva_register('diva_lakehouse.bronze', r'''"""02_Bronze: stored raw XML pages -> Bronze observations + quarantine (Delta).

* Reads the manifest and raw pages of one run, verifies each file's SHA-256.
* Keeps source values (arrays of dc values) and full traceability.
* One Bronze row = one observation of a record in one run. The observation_id
  is derived from (run_id, raw file hash, position in page), so re-processing
  the same run_id never creates duplicate observations (insert-only MERGE).
* Records without identifier / datestamp / metadata and unparseable pages go to
  the quarantine table with a reason. Nothing is dropped silently.
"""

from __future__ import annotations

import hashlib
from datetime import datetime, timezone

from .config import PipelineConfig, iso_utc, utc_now
from .ingestion import manifest_path, raw_dir
from .oai_parsing import OAIXMLError, parse_list_records
from .selection import classify_record
from .storage import Lakehouse, sha256_bytes


class RawIntegrityError(RuntimeError):
    pass


def bronze_schema():
    from pyspark.sql.types import (
        ArrayType,
        BooleanType,
        IntegerType,
        StringType,
        StructField,
        StructType,
        TimestampType,
    )

    arr = ArrayType(StringType(), containsNull=False)
    return StructType([
        StructField("observation_id", StringType(), False),
        StructField("record_id", StringType(), False),
        StructField("source_datestamp", TimestampType(), False),
        StructField("source_datestamp_raw", StringType(), False),
        StructField("set_specs", arr, False),
        StructField("deleted", BooleanType(), False),
        StructField("is_selected", BooleanType(), False),
        StructField("selection_reason", StringType(), False),
        StructField("dc_title", arr, False),
        StructField("dc_subject", arr, False),
        StructField("dc_description", arr, False),
        StructField("dc_publisher", arr, False),
        StructField("dc_date", arr, False),
        StructField("dc_type", arr, False),
        StructField("dc_identifier", arr, False),
        StructField("dc_language", arr, False),
        StructField("raw_file_path", StringType(), False),
        StructField("raw_file_sha256", StringType(), False),
        StructField("record_index", IntegerType(), False),
        StructField("fetched_at", TimestampType(), False),
        StructField("run_id", StringType(), False),
        StructField("source_mode", StringType(), False),
        StructField("endpoint", StringType(), False),
        StructField("harvest_complete", BooleanType(), False),
        StructField("ingested_at", TimestampType(), False),
    ])


def quarantine_schema():
    from pyspark.sql.types import IntegerType, StringType, StructField, StructType, TimestampType

    return StructType([
        StructField("quarantine_id", StringType(), False),
        StructField("run_id", StringType(), False),
        StructField("raw_file_path", StringType(), False),
        StructField("raw_file_sha256", StringType(), False),
        StructField("record_index", IntegerType(), False),  # -1 = whole page
        StructField("oai_identifier", StringType(), True),
        StructField("source_datestamp_raw", StringType(), True),
        StructField("error_reason", StringType(), False),
        StructField("raw_snippet", StringType(), True),
        StructField("fetched_at", TimestampType(), False),
        StructField("ingested_at", TimestampType(), False),
    ])


def _obs_id(run_id: str, sha: str, index: int) -> str:
    return hashlib.sha256(f"{run_id}|{sha}|{index}".encode()).hexdigest()


def _parse_utc(value: str) -> datetime:
    return datetime.strptime(value, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)


def build_bronze_rows(cfg: PipelineConfig, lakehouse: Lakehouse, run_id: str) -> tuple:
    """Pure Python: return (bronze_rows, quarantine_rows, stats). No Spark needed."""
    manifest = lakehouse.read_json(manifest_path(cfg, run_id))
    ingested_at = utc_now()
    rows, quarantine = [], []
    stats = {"pages": 0, "observations": 0, "selected": 0, "not_selected": 0, "deleted": 0, "quarantined": 0,
             "records_ignored_beyond_max_records": 0}
    for page in manifest["pages"]:
        rel = f"{raw_dir(cfg, run_id)}/{page['file']}"
        body = lakehouse.read_bytes(rel)
        sha = sha256_bytes(body)
        if sha != page["sha256"]:
            raise RawIntegrityError(f"SHA-256 mismatch for {rel}: manifest {page['sha256']} != file {sha}")
        stats["pages"] += 1
        fetched_at = _parse_utc(page.get("fetched_at_utc") or manifest["fetch_started_at_utc"])
        display_path = lakehouse.display(rel)
        try:
            parsed = parse_list_records(body)
        except OAIXMLError as exc:
            quarantine.append(_q(run_id, display_path, sha, -1, None, None, f"page not parseable: {exc}",
                                 body[:4000].decode("utf-8", "replace"), fetched_at, ingested_at))
            continue
        if parsed.fatal_errors:
            reason = "; ".join(f"{e.code}: {e.message}" for e in parsed.fatal_errors)
            quarantine.append(_q(run_id, display_path, sha, -1, None, None, f"OAI error page: {reason}",
                                 body[:4000].decode("utf-8", "replace"), fetched_at, ingested_at))
            continue
        use = page.get("records_used", len(parsed.records))
        stats["records_ignored_beyond_max_records"] += max(0, len(parsed.records) - use)
        for rec in parsed.records[:use]:
            if not rec.is_valid:
                quarantine.append(_q(run_id, display_path, sha, rec.index, rec.identifier, rec.datestamp_raw,
                                     "; ".join(rec.problems), rec.xml_snippet, fetched_at, ingested_at))
                continue
            is_sel, reason = classify_record(rec.dc.get("type", []), rec.deleted, cfg.thesis_filter)
            rows.append({
                "observation_id": _obs_id(run_id, sha, rec.index),
                "record_id": rec.identifier,
                "source_datestamp": rec.datestamp,
                "source_datestamp_raw": rec.datestamp_raw,
                "set_specs": list(rec.set_specs),
                "deleted": rec.deleted,
                "is_selected": is_sel,
                "selection_reason": reason,
                **{f"dc_{f}": list(rec.dc.get(f, [])) for f in
                   ("title", "subject", "description", "publisher", "date", "type", "identifier", "language")},
                "raw_file_path": display_path,
                "raw_file_sha256": sha,
                "record_index": rec.index,
                "fetched_at": fetched_at,
                "run_id": run_id,
                "source_mode": cfg.source_mode,
                "endpoint": manifest["endpoint"],
                "harvest_complete": bool(manifest["complete"]),
                "ingested_at": ingested_at,
            })
            stats["observations"] += 1
            if rec.deleted:
                stats["deleted"] += 1
            elif is_sel:
                stats["selected"] += 1
            else:
                stats["not_selected"] += 1
    stats["quarantined"] = len(quarantine)
    stats["harvest_complete"] = bool(manifest["complete"])
    stats["manifest_counts"] = manifest["counts"]
    return rows, quarantine, stats


def _q(run_id, path, sha, index, ident, ds_raw, reason, snippet, fetched_at, ingested_at):
    return {
        "quarantine_id": _obs_id(run_id, sha, index),
        "run_id": run_id,
        "raw_file_path": path,
        "raw_file_sha256": sha,
        "record_index": index,
        "oai_identifier": ident,
        "source_datestamp_raw": ds_raw,
        "error_reason": reason,
        "raw_snippet": snippet,
        "fetched_at": fetched_at,
        "ingested_at": ingested_at,
    }


def write_bronze(spark, cfg: PipelineConfig, rows: list, quarantine: list) -> dict:
    from .spark_utils import ensure_table, merge_insert_only

    t = cfg.tables
    ensure_table(spark, t["bronze"], bronze_schema())
    ensure_table(spark, t["quarantine"], quarantine_schema())
    out = {}
    if rows:
        out["bronze_rows_inserted"] = merge_insert_only(
            spark, t["bronze"], spark.createDataFrame(rows, bronze_schema()), "observation_id")
    else:
        out["bronze_rows_inserted"] = 0
    if quarantine:
        out["quarantine_rows_inserted"] = merge_insert_only(
            spark, t["quarantine"], spark.createDataFrame(quarantine, quarantine_schema()), "quarantine_id")
    else:
        out["quarantine_rows_inserted"] = 0
    out["written_at_utc"] = iso_utc(utc_now())
    return out
''')

In [ ]:
# Embedded module diva_lakehouse.silver (generated from src/diva_lakehouse/silver.py)
_diva_register('diva_lakehouse.silver', r'''"""03_Silver: current version per record_id via deterministic dedup + Delta MERGE.

Priority when several observations of the same record_id exist (in one batch
or across runs): newest source_datestamp, then newest fetched_at, then the
greatest raw_file_path, then the greatest record_index. An observation older
than what Silver already holds (source_datestamp < target) is ignored.

Actions (computed before the MERGE, from the same target snapshot):
  new             record_id not in Silver, not deleted        -> INSERT
  changed         content_hash differs                         -> UPDATE
  restored        Silver row is deleted, observation is not    -> UPDATE
  deleted         deleted header for an active Silver row      -> UPDATE is_deleted
  unchanged       same content and deletion status             -> untouched
  stale_ignored   observation older than Silver's datestamp    -> untouched
  deleted_unknown deleted header for an id never seen          -> untouched
"""

from __future__ import annotations

from .cleaning import build_silver_candidate
from .config import PipelineConfig, iso_utc, utc_now

BUSINESS_COLUMNS = (
    "title", "titles", "abstract", "abstracts", "subjects", "publication_year", "publication_year_status",
    "date_raw", "language", "language_codes", "language_raw", "publishers", "university", "university_group",
    "university_method", "university_candidates", "document_types", "dc_identifiers", "source_url", "set_specs",
)
APPLIED_ACTIONS = ("new", "changed", "restored", "deleted")
ALL_ACTIONS = ("new", "changed", "restored", "deleted", "unchanged", "stale_ignored", "deleted_unknown")


def _business_fields():
    from pyspark.sql.types import ArrayType, IntegerType, StringType, StructField

    arr = ArrayType(StringType(), containsNull=False)
    types = {
        "title": StringType(), "titles": arr, "abstract": StringType(), "abstracts": arr, "subjects": arr,
        "publication_year": IntegerType(), "publication_year_status": StringType(), "date_raw": arr,
        "language": StringType(), "language_codes": arr, "language_raw": arr, "publishers": arr,
        "university": StringType(), "university_group": StringType(), "university_method": StringType(),
        "university_candidates": arr, "document_types": arr, "dc_identifiers": arr, "source_url": StringType(),
        "set_specs": arr,
    }
    nullable = {"title", "abstract", "publication_year", "university", "source_url"}
    return [StructField(c, types[c], c in nullable) for c in BUSINESS_COLUMNS]


def silver_schema():
    from pyspark.sql.types import BooleanType, StringType, StructField, StructType, TimestampType

    return StructType(
        [StructField("record_id", StringType(), False)]
        + _business_fields()
        + [
            StructField("content_hash", StringType(), False),
            StructField("is_deleted", BooleanType(), False),
            StructField("source_datestamp", TimestampType(), False),
            StructField("source_observation_id", StringType(), False),
            StructField("source_raw_file_path", StringType(), False),
            StructField("source_raw_file_sha256", StringType(), False),
            StructField("source_run_id", StringType(), False),
            StructField("first_seen_at", TimestampType(), False),
            StructField("first_seen_run_id", StringType(), False),
            StructField("last_changed_at", TimestampType(), False),
            StructField("last_changed_run_id", StringType(), False),
        ]
    )


def candidate_schema():
    from pyspark.sql.types import BooleanType, IntegerType, StringType, StructField, StructType, TimestampType

    return StructType(
        [StructField("record_id", StringType(), False)]
        + _business_fields()
        + [
            StructField("content_hash", StringType(), False),
            StructField("deleted", BooleanType(), False),
            StructField("source_datestamp", TimestampType(), False),
            StructField("fetched_at", TimestampType(), False),
            StructField("observation_id", StringType(), False),
            StructField("raw_file_path", StringType(), False),
            StructField("raw_file_sha256", StringType(), False),
            StructField("record_index", IntegerType(), False),
            StructField("run_id", StringType(), False),
        ]
    )


def to_candidates(spark, bronze_df):
    """Bronze observations (selected or deleted) -> cleaned candidate rows with content_hash.

    Cleaning runs in plain Python on the driver: input volume is capped by
    max_records per run, and the rules stay unit-testable without Spark.
    """
    rows = []
    for r in bronze_df.where("is_selected").collect():
        obs = r.asDict(recursive=True)
        c = build_silver_candidate(obs)
        c.update({
            "deleted": obs["deleted"], "source_datestamp": obs["source_datestamp"], "fetched_at": obs["fetched_at"],
            "observation_id": obs["observation_id"], "raw_file_path": obs["raw_file_path"],
            "raw_file_sha256": obs["raw_file_sha256"], "record_index": obs["record_index"], "run_id": obs["run_id"],
        })
        rows.append(c)
    return spark.createDataFrame(rows, candidate_schema())


def deduplicate(candidates_df):
    """Keep exactly one (the highest-priority) observation per record_id."""
    from pyspark.sql import Window
    from pyspark.sql import functions as F

    w = Window.partitionBy("record_id").orderBy(
        F.col("source_datestamp").desc(), F.col("fetched_at").desc(),
        F.col("raw_file_path").desc(), F.col("record_index").desc())
    return candidates_df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")


def classify(spark, cfg: PipelineConfig, batch_df):
    from pyspark.sql import functions as F

    target = spark.table(cfg.tables["silver"]).select(
        "record_id", F.col("content_hash").alias("t_hash"), F.col("is_deleted").alias("t_deleted"),
        F.col("source_datestamp").alias("t_ds"))
    j = batch_df.join(target, "record_id", "left")
    action = (
        F.when(F.col("t_ds").isNull() & F.col("deleted"), "deleted_unknown")
        .when(F.col("t_ds").isNull(), "new")
        .when(F.col("source_datestamp") < F.col("t_ds"), "stale_ignored")
        .when(F.col("deleted") & F.col("t_deleted"), "unchanged")
        .when(F.col("deleted"), "deleted")
        .when(F.col("t_deleted"), "restored")
        .when(F.col("content_hash") != F.col("t_hash"), "changed")
        .otherwise("unchanged")
    )
    return j.withColumn("_action", action).drop("t_hash", "t_deleted", "t_ds")


def merge_into_silver(spark, cfg: PipelineConfig, classified_df) -> None:
    from delta.tables import DeltaTable

    src = classified_df.where(classified_df["_action"].isin(*APPLIED_ACTIONS))
    trace = {
        "source_datestamp": "s.source_datestamp",
        "source_observation_id": "s.observation_id",
        "source_raw_file_path": "s.raw_file_path",
        "source_raw_file_sha256": "s.raw_file_sha256",
        "source_run_id": "s.run_id",
        "last_changed_at": "s.fetched_at",
        "last_changed_run_id": "s.run_id",
    }
    content = {c: f"s.{c}" for c in BUSINESS_COLUMNS}
    content.update({"content_hash": "s.content_hash", "is_deleted": "false"})
    insert = {"record_id": "s.record_id", **content, **trace,
              "first_seen_at": "s.fetched_at", "first_seen_run_id": "s.run_id"}
    # Guard in the MERGE itself as well: never let an older observation win.
    newer = "s.source_datestamp >= t.source_datestamp"
    (
        DeltaTable.forName(spark, cfg.tables["silver"]).alias("t")
        .merge(src.alias("s"), "t.record_id = s.record_id")
        .whenMatchedUpdate(condition=f"s._action = 'deleted' AND {newer}",
                           set={"is_deleted": "true", **trace})
        .whenMatchedUpdate(condition=f"s._action IN ('changed', 'restored') AND {newer}",
                           set={**content, **trace})
        .whenNotMatchedInsert(condition="s._action = 'new'", values=insert)
        .execute()
    )


def run_silver(spark, cfg: PipelineConfig, run_id: str) -> dict:
    from pyspark.sql import functions as F

    from .spark_utils import ensure_table, last_operation_metrics

    t = cfg.tables
    ensure_table(spark, t["silver"], silver_schema())
    bronze = spark.table(t["bronze"]).where(F.col("run_id") == run_id)
    candidates = to_candidates(spark, bronze)
    n_in = candidates.count()
    batch = deduplicate(candidates)
    n_batch = batch.count()
    classified = classify(spark, cfg, batch).cache()
    counts = {a: 0 for a in ALL_ACTIONS}
    for r in classified.groupBy("_action").count().collect():
        counts[r["_action"]] = r["count"]
    merge_metrics = {}
    if sum(counts[a] for a in APPLIED_ACTIONS):
        merge_into_silver(spark, cfg, classified)
        merge_metrics = last_operation_metrics(spark, t["silver"])
    classified.unpersist()
    return {
        "run_id": run_id,
        "input_observations": n_in,
        "records_in_batch_after_dedup": n_batch,
        "duplicates_removed_in_batch": n_in - n_batch,
        "actions": counts,
        "delta_merge_metrics": merge_metrics,
        "finished_at_utc": iso_utc(utc_now()),
    }
''')

In [ ]:
# Embedded module diva_lakehouse.quality (generated from src/diva_lakehouse/quality.py)
_diva_register('diva_lakehouse.quality', r'''"""Data quality checks. BLOCKING checks stop Gold/exports; WARNINGS are reported.

Silver checks run before Gold is rebuilt; Gold checks run on the freshly
computed (not yet written) Gold DataFrames. If any blocking check fails the
previous Gold tables and exports are left untouched.
"""

from __future__ import annotations

BLOCKING = "blocking"
WARNING = "warning"


class QualityGateFailed(RuntimeError):
    def __init__(self, results):
        failed = [r["check"] for r in results if r["severity"] == BLOCKING and not r["passed"]]
        super().__init__("Blocking data quality checks failed: " + ", ".join(failed)
                         + ". Gold tables and exports were NOT replaced.")
        self.results = results


def _result(check, severity, passed, details):
    return {"check": check, "severity": severity, "passed": bool(passed), "details": details}


def _schema_signature(schema) -> list:
    return [(f.name, f.dataType.simpleString()) for f in schema.fields]


def check_silver(spark, cfg, run_stats: dict | None = None) -> list:
    from pyspark.sql import functions as F

    from .bronze import bronze_schema
    from .silver import deduplicate, silver_schema, to_candidates

    t = cfg.tables
    silver = spark.table(t["silver"])
    bronze = spark.table(t["bronze"])
    results = []

    # 1. record_id unique and non-empty
    total = silver.count()
    nulls = silver.where(F.col("record_id").isNull() | (F.trim("record_id") == "")).count()
    dups = silver.groupBy("record_id").count().where("count > 1").count()
    results.append(_result("silver_record_id_unique_not_empty", BLOCKING, nulls == 0 and dups == 0,
                           {"rows": total, "null_or_empty": nulls, "duplicated_ids": dups}))

    # 2. data types: physical schema equals the declared schema
    for name, df, expected in (("silver", silver, silver_schema()), ("bronze", bronze, bronze_schema())):
        actual, exp = _schema_signature(df.schema), _schema_signature(expected)
        results.append(_result(f"{name}_schema_types", BLOCKING, actual == exp,
                               {"mismatches": [a for a in actual if a not in exp] + [e for e in exp if e not in actual]}))
    bad_year = silver.where(F.col("publication_year").isNotNull()
                            & ~F.col("publication_year").between(1900, 2100)).count()
    bad_flags = silver.where(F.col("is_deleted").isNull() | F.col("content_hash").isNull()
                             | (F.length("content_hash") != 64)).count()
    results.append(_result("silver_value_domains", BLOCKING, bad_year == 0 and bad_flags == 0,
                           {"publication_year_out_of_range": bad_year, "bad_is_deleted_or_hash": bad_flags}))

    # 3. no older version replaced a newer one: Silver must equal the
    #    highest-priority observation of every record across ALL Bronze runs.
    latest = deduplicate(to_candidates(spark, bronze)).select(
        "record_id", F.col("content_hash").alias("l_hash"), F.col("deleted").alias("l_deleted"),
        F.col("source_datestamp").alias("l_ds"))
    cmp = silver.join(latest, "record_id", "inner")
    mismatch = cmp.where(
        (F.col("l_deleted") != F.col("is_deleted"))
        | (~F.col("l_deleted") & (F.col("l_hash") != F.col("content_hash")))
        | (F.col("source_datestamp") > F.col("l_ds"))
    )
    n_mis = mismatch.count()
    sample = [r["record_id"] for r in mismatch.select("record_id").limit(10).collect()]
    missing = latest.where(~F.col("l_deleted")).join(silver, "record_id", "left_anti").count()
    results.append(_result("silver_matches_latest_observation", BLOCKING, n_mis == 0 and missing == 0,
                           {"mismatching_records": n_mis, "sample": sample,
                            "active_latest_missing_in_silver": missing}))

    # 4. Bronze has no duplicate observations (re-runs are idempotent)
    dup_obs = bronze.groupBy("observation_id").count().where("count > 1").count()
    results.append(_result("bronze_no_duplicate_observations", BLOCKING, dup_obs == 0,
                           {"duplicated_observation_ids": dup_obs}))

    # ---- warnings
    active = silver.where(~F.col("is_deleted"))
    n_active = active.count()

    def share_missing(cond):
        n = active.where(cond).count()
        return {"missing": n, "active_records": n_active, "share": round(n / n_active, 4) if n_active else None}

    for check, cond in (
        ("warn_missing_abstract", F.col("abstract").isNull()),
        ("warn_missing_publication_year", F.col("publication_year").isNull()),
        ("warn_unidentified_university", F.col("university").isNull()),
        ("warn_missing_subjects", F.size("subjects") == 0),
    ):
        d = share_missing(cond)
        results.append(_result(check, WARNING, d["missing"] == 0, d))

    q = spark.table(t["quarantine"])
    results.append(_result("warn_quarantine", WARNING, q.count() == 0,
                           {"quarantined_total": q.count(),
                            "by_reason": {r["error_reason"]: r["count"]
                                          for r in q.groupBy("error_reason").count().limit(20).collect()}}))
    not_sel = bronze.where(~F.col("is_selected")).count()
    results.append(_result("warn_filtered_out_not_student_thesis", WARNING, True,
                           {"observations_not_selected_total": not_sel,
                            "note": "Reported, not an error: records that are not student theses are kept in "
                                    "Bronze with is_selected=false and selection_reason."}))
    if run_stats:
        silver_stats = run_stats.get("silver") or {}
        acts = silver_stats.get("actions", {})
        results.append(_result("warn_stale_or_unknown_deletes", WARNING,
                               acts.get("stale_ignored", 0) == 0 and acts.get("deleted_unknown", 0) == 0,
                               {"stale_ignored": acts.get("stale_ignored", 0),
                                "deleted_unknown": acts.get("deleted_unknown", 0)}))
        manifest = run_stats.get("manifest") or {}
        if manifest:
            results.append(_result("warn_harvest_incomplete", WARNING, bool(manifest.get("complete")),
                                   {"complete": manifest.get("complete"), "stop_reason": manifest.get("stop_reason")}))
    return results


def check_gold(spark, cfg, gold: dict) -> list:
    from pyspark.sql import functions as F

    silver = spark.table(cfg.tables["silver"])
    active = silver.where(~F.col("is_deleted")).count()
    deleted = silver.where(F.col("is_deleted")).count()
    by_year = gold["by_year"].agg(F.sum("thesis_count")).first()[0] or 0
    by_uni = gold["by_university_year"].agg(F.sum("thesis_count")).first()[0] or 0
    comp_totals = {r["records_total"] for r in gold["completeness"].select("records_total").collect()}
    results = [
        _result("gold_by_year_total_equals_active_silver", BLOCKING, by_year == active,
                {"gold_total": by_year, "silver_active": active, "silver_deleted_excluded": deleted}),
        _result("gold_by_university_year_total_equals_active_silver", BLOCKING, by_uni == active,
                {"gold_total": by_uni, "silver_active": active}),
        _result("gold_completeness_total_equals_active_silver", BLOCKING, comp_totals <= {active},
                {"records_total_values": sorted(comp_totals), "silver_active": active}),
    ]
    return results


def has_blocking_failure(results) -> bool:
    return any(r["severity"] == BLOCKING and not r["passed"] for r in results)


def summarize(results) -> dict:
    return {
        "blocking_passed": not has_blocking_failure(results),
        "blocking_failed": [r["check"] for r in results if r["severity"] == BLOCKING and not r["passed"]],
        "warnings_raised": [r["check"] for r in results if r["severity"] == WARNING and not r["passed"]],
        "checks": results,
    }
''')

In [ ]:
# Embedded module diva_lakehouse.gold (generated from src/diva_lakehouse/gold.py)
_diva_register('diva_lakehouse.gold', r'''"""04_Gold: three small aggregates (full rebuild) + JSON/CSV exports.

Grain:
  gold_theses_by_year            one row per publication_year (NULL = unknown/ambiguous year)
  gold_theses_by_university_year one row per (university_group, publication_year). Every active
                                 Silver record is counted exactly once: records matching several
                                 universities go to "Flera lärosäten (ej entydigt)", records
                                 with no identified university to "Okänt lärosäte".
  gold_metadata_completeness     one row per metric over the active Silver records

Deleted records (is_deleted = true) are excluded everywhere. Results describe
only the limited sample harvested by this project.
"""

from __future__ import annotations

import csv
import io

from .config import PipelineConfig, iso_utc, utc_now
from .storage import Lakehouse

LIVE_SCOPE_NOTE = ("Begränsat urval: endast studentuppsatser som detta projekt hämtat via OAI-PMH inom angivna "
                   "datestamp-intervall och max_records. Inte statistik över hela DiVA eller Sveriges uppsatser.")
DEMO_SCOPE_NOTE = "SYNTETISK DEMO-DATA: påhittade poster för att demonstrera pipelinen. Inte verklig statistik."

COMPLETENESS_METRICS = (
    ("title", "title IS NOT NULL"),
    ("abstract", "abstract IS NOT NULL"),
    ("subjects", "size(subjects) > 0"),
    ("publication_year", "publication_year IS NOT NULL"),
    ("identified_university", "university IS NOT NULL"),
)


def scope_note(cfg: PipelineConfig) -> str:
    return DEMO_SCOPE_NOTE if cfg.source_mode == "demo" else LIVE_SCOPE_NOTE


def build_gold(spark, cfg: PipelineConfig, run_id: str) -> dict:
    from pyspark.sql import functions as F

    active = spark.table(cfg.tables["silver"]).where(~F.col("is_deleted"))
    meta = [F.lit(scope_note(cfg)).alias("data_scope"), F.lit(cfg.source_mode).alias("source_mode"),
            F.lit(run_id).alias("built_by_run_id")]

    by_year = (active.groupBy("publication_year").agg(F.count("*").cast("long").alias("thesis_count"))
               .select("publication_year", "thesis_count", *meta).orderBy(F.col("publication_year").asc_nulls_last()))

    by_uni = (active.groupBy("university_group", "publication_year")
              .agg(F.count("*").cast("long").alias("thesis_count"))
              .select("university_group", "publication_year", "thesis_count", *meta)
              .orderBy("university_group", F.col("publication_year").asc_nulls_last()))

    total = active.count()
    rows = []
    for metric, cond in COMPLETENESS_METRICS:
        n = active.where(cond).count()
        rows.append((metric, total, n, round(n / total, 4) if total else None))
    from pyspark.sql.types import DoubleType, LongType, StringType, StructField, StructType

    comp_schema = StructType([StructField("metric", StringType(), False),
                              StructField("records_total", LongType(), False),
                              StructField("records_with_value", LongType(), False),
                              StructField("share", DoubleType(), True)])
    completeness = spark.createDataFrame(rows, comp_schema).select("*", *meta)
    return {"by_year": by_year, "by_university_year": by_uni, "completeness": completeness}


def write_gold(cfg: PipelineConfig, gold: dict) -> None:
    from .spark_utils import overwrite_table

    t = cfg.tables
    overwrite_table(gold["by_year"], t["gold_by_year"])
    overwrite_table(gold["by_university_year"], t["gold_by_university_year"])
    overwrite_table(gold["completeness"], t["gold_completeness"])


def _rows(df) -> list:
    return [r.asDict() for r in df.collect()]


def _csv(rows: list) -> str:
    if not rows:
        return ""
    buf = io.StringIO()
    w = csv.DictWriter(buf, fieldnames=list(rows[0].keys()), lineterminator="\n")
    w.writeheader()
    w.writerows(rows)
    return buf.getvalue()


def export_gold(cfg: PipelineConfig, lakehouse: Lakehouse, gold: dict, run_id: str, extra_meta: dict) -> dict:
    """Write JSON + CSV to <files_base>/exports/ via a staging folder.

    Files are first written to exports/_staging/<run_id>/ and only then moved
    into place, so a failure while writing never leaves a half-replaced set.
    """
    import os

    exports = f"{cfg.files_base}/exports"
    staging = f"{exports}/_staging/{run_id}"
    lakehouse.remove_tree(staging)
    names = {"by_year": "gold_theses_by_year", "by_university_year": "gold_theses_by_university_year",
             "completeness": "gold_metadata_completeness"}
    written = []
    for key, base in names.items():
        rows = _rows(gold[key])
        lakehouse.write_json(f"{staging}/{base}.json", rows)
        lakehouse.write_text(f"{staging}/{base}.csv", _csv(rows))
        written += [f"{base}.json", f"{base}.csv"]
    meta = {"run_id": run_id, "source_mode": cfg.source_mode, "data_scope": scope_note(cfg),
            "exported_at_utc": iso_utc(utc_now()), **extra_meta}
    lakehouse.write_json(f"{staging}/export_metadata.json", meta)
    written.append("export_metadata.json")
    for name in written:
        os.replace(lakehouse.path(f"{staging}/{name}"), lakehouse.path(f"{exports}/{name}"))
    lakehouse.remove_tree(f"{exports}/_staging")
    return {"folder": lakehouse.display(exports), "files": written}
''')

In [ ]:
# Embedded module diva_lakehouse.pipeline (generated from src/diva_lakehouse/pipeline.py)
_diva_register('diva_lakehouse.pipeline', r'''"""Stage orchestration used by the notebooks (00_Run_All and 01-04).

State between notebooks is kept in the Lakehouse Files section, so 01-04 can
run as separate notebooks/sessions in order, while 00_Run_All runs everything
in one session.
"""

from __future__ import annotations

import json
from dataclasses import dataclass, field, replace

from . import bronze as bronze_mod
from . import gold as gold_mod
from . import quality as quality_mod
from . import silver as silver_mod
from .config import ConfigError, PipelineConfig, iso_utc, new_run_id, utc_now
from .demo_data import SYNTHETIC_ENDPOINT, SyntheticTransport
from .ingestion import harvest, manifest_path
from .oai_client import OAIClient, urllib_transport
from .storage import Lakehouse


@dataclass
class RunContext:
    spark: object
    cfg: PipelineConfig
    lakehouse: Lakehouse
    run_id: str
    state: dict = field(default_factory=dict)
    log: object = print

    def rel(self, *parts) -> str:
        return "/".join([self.cfg.files_base, *parts])

    @property
    def state_dir(self) -> str:
        return self.rel("state", self.run_id)

    def save_state(self, name: str, obj) -> None:
        self.lakehouse.write_json(f"{self.state_dir}/{name}.json", obj)

    def load_state(self, name: str):
        p = f"{self.state_dir}/{name}.json"
        return self.lakehouse.read_json(p) if self.lakehouse.exists(p) else None


# ------------------------------------------------------------------ run setup
def start_run(spark, cfg: PipelineConfig, lakehouse: Lakehouse | None = None, log=print) -> RunContext:
    """Used by 00 and 01: check the Lakehouse, fix the run_id, record start time."""
    lakehouse = lakehouse or Lakehouse(cfg.lakehouse_root)
    lakehouse.check_available()
    run_id = cfg.run_id or new_run_id(cfg.source_mode)
    ctx = RunContext(spark, cfg, lakehouse, run_id, log=log)
    state = ctx.load_state("run_state")
    if state is None:
        state = {"run_id": run_id, "source_mode": cfg.source_mode, "run_mode": cfg.run_mode,
                 "started_at_utc": iso_utc(utc_now()), "config": cfg.to_dict()}
        ctx.save_state("run_state", state)
    ctx.state = state
    lakehouse.write_text(ctx.rel("state", "latest_run_id.txt"), run_id)
    log(f"Run {run_id} | mode={cfg.source_mode}/{cfg.run_mode} | tables prefix={cfg.table_prefix}_* | "
        f"files={lakehouse.display(cfg.files_base)}")
    return ctx


def resume_run(spark, cfg: PipelineConfig, lakehouse: Lakehouse | None = None, log=print) -> RunContext:
    """Used by 02-04: continue the run given by run_id, or the latest started run."""
    lakehouse = lakehouse or Lakehouse(cfg.lakehouse_root)
    lakehouse.check_available()
    run_id = cfg.run_id
    pointer = f"{cfg.files_base}/state/latest_run_id.txt"
    if not run_id:
        if not lakehouse.exists(pointer):
            raise ConfigError("No run_id given and no previous run found. Run 01_Ingestion first.")
        run_id = lakehouse.read_text(pointer).strip()
    ctx = RunContext(spark, cfg, lakehouse, run_id, log=log)
    ctx.state = ctx.load_state("run_state") or {}
    if not ctx.state:
        raise ConfigError(f"Run {run_id} has no state in {lakehouse.display(ctx.state_dir)}. Run 01_Ingestion first.")
    log(f"Continuing run {run_id} ({cfg.source_mode})")
    return ctx


# ------------------------------------------------------------------ harvest log
def _harvest_log_schema():
    from pyspark.sql.types import BooleanType, IntegerType, StringType, StructField, StructType, TimestampType

    return StructType([
        StructField("run_id", StringType(), False),
        StructField("source_mode", StringType(), False),
        StructField("endpoint", StringType(), False),
        StructField("set_spec", StringType(), False),
        StructField("from_datestamp", StringType(), True),
        StructField("until_datestamp", StringType(), True),
        StructField("complete", BooleanType(), False),
        StructField("stop_reason", StringType(), True),
        StructField("records_fetched", IntegerType(), False),
        StructField("selected_theses", IntegerType(), False),
        StructField("fetched_at", TimestampType(), False),
        StructField("processed", BooleanType(), False),  # True once Silver MERGE succeeded
    ])


def _upsert_harvest_log(ctx: RunContext, manifest: dict, processed: bool) -> None:
    from datetime import datetime, timezone

    from delta.tables import DeltaTable

    from .spark_utils import ensure_table

    name = ctx.cfg.tables["harvest_log"]
    ensure_table(ctx.spark, name, _harvest_log_schema())
    req = manifest["requested"]
    row = {
        "run_id": ctx.run_id, "source_mode": ctx.cfg.source_mode, "endpoint": manifest["endpoint"],
        "set_spec": req.get("set") or "", "from_datestamp": req.get("from"), "until_datestamp": req.get("until"),
        "complete": bool(manifest["complete"]), "stop_reason": manifest.get("stop_reason"),
        "records_fetched": int(manifest["counts"]["records_fetched"]),
        "selected_theses": int(manifest["counts"]["selected_theses"]),
        "fetched_at": datetime.strptime(manifest["fetch_started_at_utc"], "%Y-%m-%dT%H:%M:%SZ").replace(
            tzinfo=timezone.utc),
        "processed": processed,
    }
    df = ctx.spark.createDataFrame([row], _harvest_log_schema())
    (DeltaTable.forName(ctx.spark, name).alias("t").merge(df.alias("s"), "t.run_id = s.run_id")
     .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())


def resolve_from_datestamp(ctx: RunContext) -> str:
    """'auto' -> until of the last COMPLETE and PROCESSED harvest (same endpoint/set).

    Incomplete harvests (stopped at max_records/max_pages, errors) never move
    the watermark. The interval boundary day is re-requested (from = previous
    until); overlaps are harmless because Bronze/Silver are idempotent.
    """
    cfg = ctx.cfg
    if cfg.from_datestamp != "auto":
        return cfg.from_datestamp
    from .spark_utils import table_exists

    name = cfg.tables["harvest_log"]
    if not table_exists(ctx.spark, name):
        raise ConfigError("from_datestamp='auto' needs at least one completed harvest. Give an explicit date.")
    from pyspark.sql import functions as F

    row = (ctx.spark.table(name)
           .where(F.col("complete") & F.col("processed") & (F.col("endpoint") == cfg.oai_endpoint)
                  & (F.col("set_spec") == cfg.set_spec))
           .agg(F.max("until_datestamp")).first())
    if row is None or row[0] is None:
        raise ConfigError("from_datestamp='auto': no completed+processed harvest found. Give an explicit date.")
    return row[0]


# ------------------------------------------------------------------ stages
def make_client(cfg: PipelineConfig, log=print) -> OAIClient:
    if cfg.source_mode == "demo":
        return OAIClient(SYNTHETIC_ENDPOINT, transport=SyntheticTransport(cfg.demo_batch), polite_delay_s=0,
                         max_retries=0, log=log)
    return OAIClient(cfg.oai_endpoint, transport=urllib_transport, timeout_s=cfg.request_timeout_s,
                     max_retries=cfg.max_retries, backoff_base_s=cfg.backoff_base_s,
                     polite_delay_s=cfg.polite_delay_s, user_agent=cfg.user_agent, log=log)


def stage_ingestion(ctx: RunContext, client: OAIClient | None = None) -> dict:
    cfg, lh = ctx.cfg, ctx.lakehouse
    mpath = manifest_path(cfg, ctx.run_id)
    if cfg.reuse_existing_raw and lh.exists(mpath):
        manifest = lh.read_json(mpath)
        ctx.log(f"Raw data for run {ctx.run_id} already exists; reusing {lh.display(mpath)} (no new requests).")
    else:
        if cfg.source_mode == "demo":
            from_ds, until_ds = None, None
        else:
            from_ds, until_ds = resolve_from_datestamp(ctx), cfg.until_datestamp
            if from_ds > until_ds:
                raise ConfigError(f"Resolved from_datestamp {from_ds} is after until_datestamp {until_ds}.")
        client = client or make_client(cfg, ctx.log)
        ctx.log(f"Harvesting from={from_ds} until={until_ds} set={cfg.set_spec or '-'} max_records={cfg.max_records}")
        manifest = harvest(cfg, lh, client, ctx.run_id, from_ds, until_ds, log=ctx.log)
    _upsert_harvest_log(ctx, manifest, processed=False)
    c = manifest["counts"]
    ctx.log(f"Ingestion: complete={manifest['complete']} ({manifest['stop_reason']}), pages={c['pages']}, "
            f"fetched={c['records_fetched']}, selected={c['selected_theses']}, deleted={c['deleted_headers']}, "
            f"not_selected={c['not_selected']}, invalid={c['invalid_records']}")
    return manifest


def stage_bronze(ctx: RunContext) -> dict:
    rows, quarantine, stats = bronze_mod.build_bronze_rows(ctx.cfg, ctx.lakehouse, ctx.run_id)
    stats.update(bronze_mod.write_bronze(ctx.spark, ctx.cfg, rows, quarantine))
    ctx.save_state("bronze", stats)
    ctx.log(f"Bronze: observations={stats['observations']} (inserted {stats['bronze_rows_inserted']}), "
            f"quarantined={stats['quarantined']} (inserted {stats['quarantine_rows_inserted']})")
    return stats


def stage_silver(ctx: RunContext) -> dict:
    stats = silver_mod.run_silver(ctx.spark, ctx.cfg, ctx.run_id)
    ctx.save_state("silver", stats)
    manifest = ctx.lakehouse.read_json(manifest_path(ctx.cfg, ctx.run_id))
    _upsert_harvest_log(ctx, manifest, processed=True)
    ctx.log("Silver: " + ", ".join(f"{k}={v}" for k, v in stats["actions"].items())
            + f" | duplicates removed in batch={stats['duplicates_removed_in_batch']}")
    return stats


def stage_quality_gold(ctx: RunContext) -> dict:
    """Blocking checks -> Gold (computed) -> Gold checks -> write Gold + exports -> report."""
    cfg, spark = ctx.cfg, ctx.spark
    manifest = ctx.lakehouse.read_json(manifest_path(cfg, ctx.run_id))
    run_stats = {"manifest": manifest, "silver": ctx.load_state("silver"), "bronze": ctx.load_state("bronze")}
    results = quality_mod.check_silver(spark, cfg, run_stats)
    gold_info = {"written": False}
    exports = None
    if not quality_mod.has_blocking_failure(results):
        gold = gold_mod.build_gold(spark, cfg, ctx.run_id)
        results += quality_mod.check_gold(spark, cfg, gold)
        if not quality_mod.has_blocking_failure(results):
            gold_mod.write_gold(cfg, gold)
            gold_info = {"written": True, "row_counts": {k: gold[k].count() for k in gold}}
            exports = gold_mod.export_gold(cfg, ctx.lakehouse, gold, ctx.run_id,
                                           {"harvested_intervals": harvested_intervals(ctx)})
    summary = quality_mod.summarize(results)
    ctx.save_state("quality", summary)
    report = write_run_report(ctx, manifest, run_stats, summary, gold_info, exports)
    print_quality(summary, ctx.log)
    if not summary["blocking_passed"]:
        raise quality_mod.QualityGateFailed(results)
    return report


def harvested_intervals(ctx: RunContext) -> list:
    rows = ctx.spark.table(ctx.cfg.tables["harvest_log"]).orderBy("fetched_at").collect()
    return [{"run_id": r["run_id"], "from": r["from_datestamp"], "until": r["until_datestamp"],
             "set": r["set_spec"], "complete": r["complete"], "processed": r["processed"],
             "records_fetched": r["records_fetched"]} for r in rows]


def write_run_report(ctx, manifest, run_stats, quality_summary, gold_info, exports) -> dict:
    silver = run_stats.get("silver") or {}
    bronze = run_stats.get("bronze") or {}
    acts = silver.get("actions", {})
    report = {
        "run_id": ctx.run_id,
        "source_mode": ctx.cfg.source_mode,
        "run_mode": ctx.cfg.run_mode,
        "data_label": gold_mod.scope_note(ctx.cfg),
        "started_at_utc": ctx.state.get("started_at_utc"),
        "finished_at_utc": iso_utc(utc_now()),
        "status": "succeeded" if quality_summary["blocking_passed"] else "failed_blocking_quality_checks",
        "harvest": {
            "endpoint": manifest["endpoint"],
            "requested": manifest["requested"],
            "complete": manifest["complete"],
            "stop_reason": manifest["stop_reason"],
            "counts": manifest["counts"],
            "notes": manifest.get("notes", []),
        },
        "records": {
            "fetched": manifest["counts"]["records_fetched"],
            "accepted_into_bronze": bronze.get("observations"),
            "selected_student_theses": bronze.get("selected"),
            "not_selected": bronze.get("not_selected"),
            "deleted_headers": bronze.get("deleted"),
            "quarantined": bronze.get("quarantined"),
        },
        "silver_changes": {
            "new": acts.get("new", 0),
            "changed": acts.get("changed", 0) + acts.get("restored", 0),
            "unchanged": acts.get("unchanged", 0),
            "deleted": acts.get("deleted", 0),
            "stale_ignored": acts.get("stale_ignored", 0),
            "deleted_unknown": acts.get("deleted_unknown", 0),
            "duplicates_removed_in_batch": silver.get("duplicates_removed_in_batch", 0),
        },
        "quality": {k: quality_summary[k] for k in ("blocking_passed", "blocking_failed", "warnings_raised")},
        "quality_checks": quality_summary["checks"],
        "gold": gold_info,
        "exports": exports,
    }
    path = ctx.rel("reports", f"{ctx.run_id}.json")
    ctx.lakehouse.write_json(path, report)
    report["report_path"] = ctx.lakehouse.display(path)
    ctx.log(f"Run report: {report['report_path']} (status={report['status']})")
    return report


def print_quality(summary: dict, log=print) -> None:
    for r in summary["checks"]:
        mark = "PASS" if r["passed"] else ("FAIL" if r["severity"] == "blocking" else "WARN")
        log(f"  [{mark}] {r['severity']:8s} {r['check']}: {json.dumps(r['details'], ensure_ascii=False, default=str)}")


def run_pipeline(spark, cfg: PipelineConfig, lakehouse: Lakehouse | None = None, log=print,
                 client: OAIClient | None = None) -> dict:
    ctx = start_run(spark, cfg, lakehouse, log)
    stage_ingestion(ctx, client)
    stage_bronze(ctx)
    stage_silver(ctx)
    return stage_quality_gold(ctx)


# ------------------------------------------------------------------ re-run demo
def _snapshot(spark, cfg: PipelineConfig) -> dict:
    def rows(name, drop=()):
        df = spark.table(name).drop(*drop)
        return sorted(json.dumps(r.asDict(recursive=True), sort_keys=True, default=str) for r in df.collect())

    t = cfg.tables
    return {
        "silver": rows(t["silver"]),
        "gold_by_year": rows(t["gold_by_year"], ("built_by_run_id",)),
        "gold_by_university_year": rows(t["gold_by_university_year"], ("built_by_run_id",)),
        "gold_completeness": rows(t["gold_completeness"], ("built_by_run_id",)),
    }


def reset_demo(spark, cfg: PipelineConfig, lakehouse: Lakehouse) -> None:
    from .spark_utils import drop_table

    if cfg.source_mode != "demo" or "demo" not in cfg.table_prefix or "demo" not in cfg.files_base:
        raise ConfigError("Refusing to reset: not a demo configuration.")
    for name in cfg.tables.values():
        drop_table(spark, name)
    lakehouse.remove_tree(cfg.files_base)


def run_rerun_demo(spark, cfg: PipelineConfig, lakehouse: Lakehouse | None = None, log=print) -> dict:
    """SYNTHETIC demo: A -> B -> B again (new run_id) -> re-process run demo-B (same run_id).

    Verifies that Silver and Gold are logically identical after steps 2, 3, 4
    and that re-processing the same run_id adds no Bronze observations.
    """
    from pyspark.sql import functions as F

    from .demo_data import EXPECTED_AFTER_B

    lakehouse = lakehouse or Lakehouse(cfg.lakehouse_root)
    lakehouse.check_available()
    reset_demo(spark, cfg, lakehouse)
    t = cfg.tables
    steps = [("1_batch_A", "A", "demo-A"), ("2_batch_B", "B", "demo-B"), ("3_batch_B_again", "B", "demo-B-repeat")]
    snaps, out = {}, {"label": "SYNTHETIC DEMO DATA", "steps": {}}
    for step, batch, run_id in steps:
        log(f"\n=== Demo step {step}: batch {batch}, run_id={run_id} ===")
        step_cfg = replace(cfg, run_id=run_id, demo_batch=batch, run_mode="rerun_demo")
        report = run_pipeline(spark, step_cfg, lakehouse, log)
        snaps[step] = _snapshot(spark, cfg)
        out["steps"][step] = {"run_id": run_id, "silver_changes": report["silver_changes"],
                              "bronze_rows_total": spark.table(t["bronze"]).count(),
                              "status": report["status"]}

    log("\n=== Demo step 4_reprocess_same_run_id: Bronze+Silver+Gold for run_id=demo-B again ===")
    step_cfg = replace(cfg, run_id="demo-B", demo_batch="B", run_mode="rerun_demo")
    ctx = resume_run(spark, step_cfg, lakehouse, log)
    stage_ingestion(ctx)  # reuses stored raw pages, no new requests
    stage_bronze(ctx)
    stage_silver(ctx)
    report = stage_quality_gold(ctx)
    snaps["4_reprocess_same_run_id"] = _snapshot(spark, cfg)
    out["steps"]["4_reprocess_same_run_id"] = {"run_id": "demo-B", "silver_changes": report["silver_changes"],
                                               "bronze_rows_total": spark.table(t["bronze"]).count(),
                                               "status": report["status"]}

    s2, s3, s4 = snaps["2_batch_B"], snaps["3_batch_B_again"], snaps["4_reprocess_same_run_id"]
    silver = spark.table(t["silver"])
    b_changes = out["steps"]["2_batch_B"]["silver_changes"]
    checks = {
        "silver_identical_after_steps_2_3_4": s2["silver"] == s3["silver"] == s4["silver"],
        "gold_identical_after_steps_2_3_4": all(s2[k] == s3[k] == s4[k] for k in s2 if k.startswith("gold")),
        "bronze_unchanged_when_same_run_id_reprocessed":
            out["steps"]["3_batch_B_again"]["bronze_rows_total"]
            == out["steps"]["4_reprocess_same_run_id"]["bronze_rows_total"],
        "step3_and_4_report_no_changes": all(
            out["steps"][s]["silver_changes"][k] == 0 for s in ("3_batch_B_again", "4_reprocess_same_run_id")
            for k in ("new", "changed", "deleted")),
        "batch_B_changes_as_designed": all(b_changes[k] == EXPECTED_AFTER_B[k]
                                           for k in ("new", "changed", "unchanged", "deleted")),
        "silver_rows_as_designed": silver.count() == EXPECTED_AFTER_B["silver_rows"]
        and silver.where(~F.col("is_deleted")).count() == EXPECTED_AFTER_B["silver_active"],
        "no_duplicate_record_ids": silver.groupBy("record_id").count().where("count > 1").count() == 0,
    }
    out["checks"] = checks
    out["passed"] = all(checks.values())
    out["finished_at_utc"] = iso_utc(utc_now())
    path = f"{cfg.files_base}/reports/rerun_demo_result.json"
    lakehouse.write_json(path, out)
    log("\n=== Re-run demo result (SYNTHETIC) ===")
    for k, v in checks.items():
        log(f"  [{'PASS' if v else 'FAIL'}] {k}")
    log(f"Saved: {lakehouse.display(path)}")
    if not out["passed"]:
        raise AssertionError("Re-run demo checks failed: " + ", ".join(k for k, v in checks.items() if not v))
    return out
''')

In [ ]:
from diva_lakehouse import pipeline as P
from diva_lakehouse.config import PipelineConfig

cfg = PipelineConfig(
    source_mode=source_mode,
    oai_endpoint=oai_endpoint,
    from_datestamp=from_datestamp,
    until_datestamp=until_datestamp,
    set_spec=set_spec,
    max_records=int(max_records),
    table_prefix=table_prefix,
    run_id=run_id,
    run_mode=run_mode,
    demo_batch=demo_batch,
)
print("Tables:", cfg.tables)

In [ ]:
if cfg.run_mode != "pipeline":
    raise ValueError("run_mode='rerun_demo' is only supported in 00_Run_All.")
ctx = P.start_run(spark, cfg)
manifest = P.stage_ingestion(ctx)